<a href="https://colab.research.google.com/github/rick-starnez/simulador-spice-modo_mixto/blob/main/SimuladorSPICE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files
import os

# --- Estado de la aplicación ---
app_state = {
    "current_file": None,
    "editor_content": ""
}

# --- Widgets de la GUI ---
editor = widgets.Textarea(
    value='* Circuito de ejemplo\nV1 1 0 DC 5\nR1 1 2 1k\nR2 2 0 1k\n.op\n.end',
    placeholder='Escribe tu netlist SPICE aquí...',
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='300px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt',
    multiple=False,
    description='Subir archivo SPICE'
)

# --- Funciones de los botones ---
def on_nuevo(b):
    editor.value = ''
    app_state["current_file"] = None
    with output_area:
        clear_output()
        print("Editor limpio. Listo para un nuevo circuito.")

def on_abrir(b):
    # Mostramos el widget de subida
    display(uploader)

def on_file_uploaded(change):
    if not change.new:
        return
    # En Colab, el valor es un diccionario
    for filename, fileinfo in change.new.items():
        content = fileinfo['content'].decode('utf-8')
        editor.value = content
        app_state["current_file"] = filename
        with output_area:
            clear_output()
            print(f"Archivo cargado: {filename}")
    # Ocultamos el uploader después de usarlo
    uploader.layout.display = 'none'

def on_guardar(b):
    # Guardado temporal en el disco de Colab
    if app_state["current_file"]:
        filename = app_state["current_file"]
    else:
        filename = 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f:
        f.write(editor.value)
    with output_area:
        clear_output()
        print(f"Guardado en el entorno de Colab: /content/{filename}")
        print("(Recuerda que este archivo se pierde al cerrar la sesión. Usa 'Guardar como' para descargarlo.)")

def on_guardar_como(b):
    # Descarga directa al ordenador local
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f:
        f.write(editor.value)
    colab_files.download(f'/content/{filename}')
    with output_area:
        clear_output()
        print(f"Descargando {filename} a tu ordenador...")

def on_simular(b):
    with output_area:
        clear_output()
        print("Simulando... (Aquí iría la llamada a PySpice o tu motor SPICE)")
        # Aquí es donde integrarías PySpice. Ejemplo conceptual:
        # from pyspice... import ...
        # analysis = simular(editor.value)
        # print(analysis)
        print("\n--- Netlist a simular ---")
        print(editor.value)

# --- Conectar eventos ---
btn_nuevo.on_click(on_nuevo)
btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar)
btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

# --- Layout ---
barra_botones = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
# Ocultamos el uploader hasta que se pulse "Abrir"
uploader.layout.display = 'none'

display(barra_botones, editor, output_area)

Textarea(value='* Circuito de ejemplo\nV1 1 0 DC 5\nR1 1 2 1k\nR2 2 0 1k\n.op\n.end', description='Netlist:', …

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python)
#  Version 3: Consolidada con Co-simulacion, PWL y Graficas
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    c_code = f"""
    #include <math.h>
    void ejecutar_c(double *entradas, double *salidas) {{
    """
    for i, p in enumerate(puertos_in):
        c_code += f"    double {p} = entradas[{i}];\n"
    for p in puertos_out:
        c_code += f"    double {p} = 0.0;\n"

    c_code += codigo + "\n"

    for i, p in enumerate(puertos_out):
        c_code += f"    salidas[{i}] = {p};\n"
    c_code += "}\n"

    c_path = f"/content/{nombre}.c"
    so_path = f"/content/{nombre}.so"
    with open(c_path, 'w') as f:
        f.write(c_code)

    subprocess.run(['gcc', '-shared', '-fPIC', '-o', so_path, c_path, '-lm'], check=True)

    lib = ctypes.CDLL(so_path)
    lib.ejecutar_c.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
    return lib.ejecutar_c

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl:
            self.fuentes_pwl[nombre] = pwl

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        nodos_dig = [mapeo[p] for p in pins_dig if p in mapeo]
        nodos_analog = [mapeo[p] for p in pins_analog if p in mapeo]
        nodos_out = [mapeo[p] for p in pins_out if p in mapeo]

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        func_compilada = None
        if lang.lower() == 'c':
            func_compilada = compilar_c_y_cargar(nombre, codigo, pins_dig + pins_analog, pins_out)

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': pins_dig,
            'analog_vars': pins_analog,
            'out_vars': pins_out,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'func': func_compilada,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in pins_out}
        })

    def _estampar_bloques(self, A, b, idx, modo='dc'):
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

            g_out = 1.0 / bloque['r_out']
            for p in bloque['out_vars']:
                if p not in bloque['estado_anterior']: continue
                nodo_out = bloque['out_nodos'][bloque['out_vars'].index(p)]
                i_node = idx.get(nodo_out)
                if i_node is not None:
                    A[i_node, i_node] += g_out
                    estado = bloque['estado_anterior'][p]
                    v_pin = bloque['v_high'] if estado > 0.5 else bloque['v_low']
                    b[i_node] += v_pin * g_out

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        self._estampar_bloques(A, b, idx, modo)

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = []
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                val_logico = 1.0 if v_actual > bloque['v_th'] else 0.0
                entradas.append(val_logico)
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas.append(v_actual)

            if bloque['lang'] == 'python':
                local_env = {}
                todas_in = bloque['dig_vars'] + bloque['analog_vars']
                for i, p in enumerate(todas_in):
                    local_env[p] = entradas[i]
                for p in bloque['out_vars']:
                    local_env[p] = 0.0

                for k, v in bloque['params'].items():
                    local_env[k.lower()] = v
                local_env['v_high'] = bloque['v_high']
                local_env['v_low'] = bloque['v_low']
                local_env['r_out'] = bloque['r_out']
                local_env['r_in'] = bloque['r_in']
                local_env['v_th'] = bloque['v_th']

                try:
                    exec(bloque['codigo'], {}, local_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(local_env[p])
                except Exception as e:
                    print(f"Error en bloque Python {bloque['nombre']}: {e}")
            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*entradas)
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': []}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                linea_clean = linea.replace('->', ' ')
                tokens_clean = linea_clean.split()
                if len(tokens_clean) >= 3:
                    var = tokens_clean[1]
                    try:
                        nodo = int(tokens_clean[2])
                        subckt_mapeo[var] = nodo
                    except ValueError:
                        errores.append(f"L{n_linea}: Nodo invalido en MAP")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                    if match:
                        try:
                            nodo = int(match.group(1))
                            directivas['plot'].append(nodo)
                        except ValueError:
                            pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and 'PWL' in linea.upper():
                    match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pwl:
                        raise ValueError("Formato PWL invalido (faltan parentesis)")
                    contenido = match_pwl.group(1).strip()
                    pwl_tokens = contenido.replace(',', ' ').split()
                    pwl_puntos = []
                    for i in range(0, len(pwl_tokens), 2):
                        if i+1 < len(pwl_tokens):
                            t_val = sufijo_a_valor(pwl_tokens[i])
                            v_val = sufijo_a_valor(pwl_tokens[i+1])
                            pwl_puntos.append((t_val, v_val))
                    if not pwl_puntos:
                        raise ValueError("PWL sin puntos")
                    c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Triangulo de 0 a 5V y de regreso en 20ms\n'
        'V1 2 0 PWL(0 0 10m 5 20m 0)\n'
        '\n'
        '.SUBCKT adc_4bits\n'
        '.PORTS IN_ANALOG: sensor\n'
        '.PORTS OUT: bit0 bit1 bit2 bit3\n'
        '.MAP sensor -> 2\n'
        '.MAP bit0 -> 3\n'
        '.MAP bit1 -> 4\n'
        '.MAP bit2 -> 5\n'
        '.MAP bit3 -> 6\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG python\n'
        '.CODE\n'
        'valor = int(round((sensor / v_high) * 15.0))\n'
        'if valor > 15: valor = 15\n'
        'if valor < 0: valor = 0\n'
        'bit0 = float((valor >> 0) & 1)\n'
        'bit1 = float((valor >> 1) & 1)\n'
        'bit2 = float((valor >> 2) & 1)\n'
        'bit3 = float((valor >> 3) & 1)\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        'R_0 3 7 16k\n'
        'R_1 4 7 8k\n'
        'R_2 5 7 4k\n'
        'R_3 6 7 2k\n'
        'R_f 7 8 1k\n'
        '\n'
        'E_op 8 0 0 7 1e6\n'
        '\n'
        'R_inv1 8 9 1k\n'
        'R_inv2 9 10 1k\n'
        'E_inv 10 0 0 9 1e6\n'
        '\n'
        '.plot V(2) V(10)\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                fig, ax = plt.subplots(figsize=(10, 5))

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                for n in nodos_a_graficar:
                    if n in historial and n != 0:
                        ax.plot(tiempos, historial[n], label=f'V({n})')

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"
                ax.set_title(titulo)

                ax.set_xlabel('Time [s]')
                ax.set_ylabel('Voltage [V]')
                ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt', description='Subir archivo SPICE', layout=Layout(display='no…

Textarea(value='* Triangulo de 0 a 5V y de regreso en 20ms\nV1 2 0 PWL(0 0 10m 5 20m 0)\n\n.SUBCKT adc_4bits\n…

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 4: Soporte para VHDL Conductual basico (sin process)
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    c_code = f"""
    #include <math.h>
    void ejecutar_c(double *entradas, double *salidas) {{
    """
    for i, p in enumerate(puertos_in):
        c_code += f"    double {p} = entradas[{i}];\n"
    for p in puertos_out:
        c_code += f"    double {p} = 0.0;\n"

    c_code += codigo + "\n"

    for i, p in enumerate(puertos_out):
        c_code += f"    salidas[{i}] = {p};\n"
    c_code += "}\n"

    c_path = f"/content/{nombre}.c"
    so_path = f"/content/{nombre}.so"
    with open(c_path, 'w') as f:
        f.write(c_code)

    subprocess.run(['gcc', '-shared', '-fPIC', '-o', so_path, c_path, '-lm'], check=True)

    lib = ctypes.CDLL(so_path)
    lib.ejecutar_c.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
    return lib.ejecutar_c

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl:
            self.fuentes_pwl[nombre] = pwl

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        nodos_dig = [mapeo[p] for p in pins_dig if p in mapeo]
        nodos_analog = [mapeo[p] for p in pins_analog if p in mapeo]
        nodos_out = [mapeo[p] for p in pins_out if p in mapeo]

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        func_compilada = None
        if lang.lower() == 'c':
            func_compilada = compilar_c_y_cargar(nombre, codigo, pins_dig + pins_analog, pins_out)

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': pins_dig,
            'analog_vars': pins_analog,
            'out_vars': pins_out,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'func': func_compilada,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in pins_out}
        })

    def _estampar_bloques(self, A, b, idx, modo='dc'):
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

            g_out = 1.0 / bloque['r_out']
            for p in bloque['out_vars']:
                if p not in bloque['estado_anterior']: continue
                nodo_out = bloque['out_nodos'][bloque['out_vars'].index(p)]
                i_node = idx.get(nodo_out)
                if i_node is not None:
                    A[i_node, i_node] += g_out
                    estado = bloque['estado_anterior'][p]
                    v_pin = bloque['v_high'] if estado > 0.5 else bloque['v_low']
                    b[i_node] += v_pin * g_out

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        self._estampar_bloques(A, b, idx, modo)

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                local_env = {}
                for k, v in entradas.items(): local_env[k] = v
                for p in bloque['out_vars']: local_env[p] = 0.0

                for k, v in bloque['params'].items(): local_env[k.lower()] = v
                local_env['v_high'] = bloque['v_high']
                local_env['v_low'] = bloque['v_low']
                local_env['r_out'] = bloque['r_out']
                local_env['r_in'] = bloque['r_in']
                local_env['v_th'] = bloque['v_th']

                try:
                    exec(bloque['codigo'], {}, local_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(local_env[p])
                except Exception as e:
                    print(f"Error en bloque Python {bloque['nombre']}: {e}")

            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*[entradas[k] for k in bloque['dig_vars'] + bloque['analog_vars']])
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

            elif bloque['lang'] == 'vhdl':
                # Interprete VHDL Conductual (Asignaciones concurrentes)
                res = {}
                for p in bloque['out_vars']: res[p] = 0.0

                # Limpiamos el codigo de comentarios
                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                # Dividimos por punto y coma
                sentencias = [s.strip() for s in codigo_limpio.split(';') if s.strip()]

                for sent in sentencias:
                    sent = sent.replace("'", "") # Quitar comillas simples de VHDL ('1', '0')
                    sent = re.sub(r'\bAND\b', ' and ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bOR\b', ' or ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bNOT\b', ' not ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bXOR\b', ' ^ ', sent, flags=re.IGNORECASE)

                    # Formato: salida <= condicion WHEN expr ELSE expr
                    match_when = re.match(r'(\w+)\s*<=\s*(.*?)\s+WHEN\s+(.*?)\s+ELSE\s+(.*)', sent, re.IGNORECASE)
                    if match_when:
                        out_var = match_when.group(1).strip()
                        val_true = match_when.group(2).strip()
                        cond = match_when.group(3).strip()
                        val_false = match_when.group(4).strip()

                        local_env = dict(entradas)
                        try:
                            # Convertir 1 y 0 a True/False para evaluacion
                            cond_eval = cond.replace(' = ', ' == ')
                            if eval(cond_eval, {}, local_env):
                                res[out_var] = float(val_true)
                            else:
                                res[out_var] = float(val_false)
                        except:
                            pass
                    else:
                        # Formato simple: salida <= expr
                        match_simple = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match_simple:
                            out_var = match_simple.group(1).strip()
                            expr = match_simple.group(2).strip()
                            local_env = dict(entradas)
                            try:
                                res[out_var] = float(eval(expr, {}, local_env))
                            except:
                                pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res[p]

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': []}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                linea_clean = linea.replace('->', ' ')
                tokens_clean = linea_clean.split()
                if len(tokens_clean) >= 3:
                    var = tokens_clean[1]
                    try:
                        nodo = int(tokens_clean[2])
                        subckt_mapeo[var] = nodo
                    except ValueError:
                        errores.append(f"L{n_linea}: Nodo invalido en MAP")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                    if match:
                        try:
                            nodo = int(match.group(1))
                            directivas['plot'].append(nodo)
                        except ValueError:
                            pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and 'PWL' in linea.upper():
                    match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pwl:
                        raise ValueError("Formato PWL invalido (faltan parentesis)")
                    contenido = match_pwl.group(1).strip()
                    pwl_tokens = contenido.replace(',', ' ').split()
                    pwl_puntos = []
                    for i in range(0, len(pwl_tokens), 2):
                        if i+1 < len(pwl_tokens):
                            t_val = sufijo_a_valor(pwl_tokens[i])
                            v_val = sufijo_a_valor(pwl_tokens[i+1])
                            pwl_puntos.append((t_val, v_val))
                    if not pwl_puntos:
                        raise ValueError("PWL sin puntos")
                    c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Compuerta AND en VHDL Conductual\n'
        '* Entradas: A (nodo 2), B (nodo 3)\n'
        '* Salida: Y (nodo 4)\n'
        'VA 2 0 PWL(0 0 5m 5 10m 0 15m 5 20m 0)\n'
        'VB 3 0 PWL(0 0 10m 5 20m 0)\n'
        '\n'
        '.SUBCKT and_gate\n'
        '.PORTS IN_DIG: A B\n'
        '.PORTS OUT: Y\n'
        '.MAP A -> 2\n'
        '.MAP B -> 3\n'
        '.MAP Y -> 4\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Y <= A AND B;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        'R_load 4 0 1k\n'
        '\n'
        '.plot V(2) V(3) V(4)\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                fig, ax = plt.subplots(figsize=(10, 5))

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                for n in nodos_a_graficar:
                    if n in historial and n != 0:
                        ax.plot(tiempos, historial[n], label=f'V({n})')

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"
                ax.set_title(titulo)

                ax.set_xlabel('Time [s]')
                ax.set_ylabel('Voltage [V]')
                ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Compuerta AND en VHDL Conductual\n* Entradas: A (nodo 2), B (nodo 3)\n* Salida: Y (nodo 4)\n…

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 6: ADC (0-1V) -> VHDL (Shift Left) -> DAC Spice
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    c_code = f"""
    #include <math.h>
    void ejecutar_c(double *entradas, double *salidas) {{
    """
    for i, p in enumerate(puertos_in):
        c_code += f"    double {p} = entradas[{i}];\n"
    for p in puertos_out:
        c_code += f"    double {p} = 0.0;\n"

    c_code += codigo + "\n"

    for i, p in enumerate(puertos_out):
        c_code += f"    salidas[{i}] = {p};\n"
    c_code += "}\n"

    c_path = f"/content/{nombre}.c"
    so_path = f"/content/{nombre}.so"
    with open(c_path, 'w') as f:
        f.write(c_code)

    subprocess.run(['gcc', '-shared', '-fPIC', '-o', so_path, c_path, '-lm'], check=True)

    lib = ctypes.CDLL(so_path)
    lib.ejecutar_c.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
    return lib.ejecutar_c

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl:
            self.fuentes_pwl[nombre] = pwl

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        nodos_dig = [mapeo[p] for p in pins_dig if p in mapeo]
        nodos_analog = [mapeo[p] for p in pins_analog if p in mapeo]
        nodos_out = [mapeo[p] for p in pins_out if p in mapeo]

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        func_compilada = None
        if lang.lower() == 'c':
            func_compilada = compilar_c_y_cargar(nombre, codigo, pins_dig + pins_analog, pins_out)

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': pins_dig,
            'analog_vars': pins_analog,
            'out_vars': pins_out,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'func': func_compilada,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in pins_out}
        })

    def _estampar_bloques(self, A, b, idx, modo='dc'):
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

            g_out = 1.0 / bloque['r_out']
            for p in bloque['out_vars']:
                if p not in bloque['estado_anterior']: continue
                nodo_out = bloque['out_nodos'][bloque['out_vars'].index(p)]
                i_node = idx.get(nodo_out)
                if i_node is not None:
                    A[i_node, i_node] += g_out
                    estado = bloque['estado_anterior'][p]
                    v_pin = bloque['v_high'] if estado > 0.5 else bloque['v_low']
                    b[i_node] += v_pin * g_out

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        self._estampar_bloques(A, b, idx, modo)

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                local_env = {}
                for k, v in entradas.items(): local_env[k] = v
                for p in bloque['out_vars']: local_env[p] = 0.0

                for k, v in bloque['params'].items(): local_env[k.lower()] = v
                local_env['v_high'] = bloque['v_high']
                local_env['v_low'] = bloque['v_low']
                local_env['r_out'] = bloque['r_out']
                local_env['r_in'] = bloque['r_in']
                local_env['v_th'] = bloque['v_th']

                try:
                    exec(bloque['codigo'], {}, local_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(local_env[p])
                except Exception as e:
                    print(f"Error en bloque Python {bloque['nombre']}: {e}")

            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*[entradas[k] for k in bloque['dig_vars'] + bloque['analog_vars']])
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

            elif bloque['lang'] == 'vhdl':
                res = {}
                for p in bloque['out_vars']: res[p] = 0.0

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                sentencias = [s.strip() for s in codigo_limpio.split(';') if s.strip()]

                for sent in sentencias:
                    sent = sent.replace("'", "")
                    sent = re.sub(r'\bAND\b', ' and ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bOR\b', ' or ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bNOT\b', ' not ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bXOR\b', ' ^ ', sent, flags=re.IGNORECASE)

                    match_when = re.match(r'(\w+)\s*<=\s*(.*?)\s+WHEN\s+(.*?)\s+ELSE\s+(.*)', sent, re.IGNORECASE)
                    if match_when:
                        out_var = match_when.group(1).strip()
                        val_true = match_when.group(2).strip()
                        cond = match_when.group(3).strip()
                        val_false = match_when.group(4).strip()

                        local_env = dict(entradas)
                        try:
                            cond_eval = cond.replace(' = ', ' == ')
                            v_true = float(eval(val_true, {}, local_env))
                            v_false = float(eval(val_false, {}, local_env))
                            if eval(cond_eval, {}, local_env):
                                res[out_var] = v_true
                            else:
                                res[out_var] = v_false
                        except:
                            pass
                    else:
                        match_simple = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match_simple:
                            out_var = match_simple.group(1).strip()
                            expr = match_simple.group(2).strip()
                            local_env = dict(entradas)
                            try:
                                res[out_var] = float(eval(expr, {}, local_env))
                            except:
                                pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res[p]

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': []}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                linea_clean = linea.replace('->', ' ')
                tokens_clean = linea_clean.split()
                if len(tokens_clean) >= 3:
                    var = tokens_clean[1]
                    try:
                        nodo = int(tokens_clean[2])
                        subckt_mapeo[var] = nodo
                    except ValueError:
                        errores.append(f"L{n_linea}: Nodo invalido en MAP")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                    if match:
                        try:
                            nodo = int(match.group(1))
                            directivas['plot'].append(nodo)
                        except ValueError:
                            pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and 'PWL' in linea.upper():
                    match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pwl:
                        raise ValueError("Formato PWL invalido (faltan parentesis)")
                    contenido = match_pwl.group(1).strip()
                    pwl_tokens = contenido.replace(',', ' ').split()
                    pwl_puntos = []
                    for i in range(0, len(pwl_tokens), 2):
                        if i+1 < len(pwl_tokens):
                            t_val = sufijo_a_valor(pwl_tokens[i])
                            v_val = sufijo_a_valor(pwl_tokens[i+1])
                            pwl_puntos.append((t_val, v_val))
                    if not pwl_puntos:
                        raise ValueError("PWL sin puntos")
                    c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Sistema Mixto: Entrada (0-1V) -> ADC (Python) -> VHDL (Shift Left) -> DAC (Spice)\n'
        'V_in 1 0 PWL(0 0 10m 1 20m 0)\n' # Rampa de 0 a 1V
        '\n'
        '.SUBCKT adc_python\n'
        '.PORTS IN_ANALOG: v_analog\n'
        '.PORTS OUT: b0 b1\n'  # Solo 2 bits porque el maximo es 3 (binario 11)
        '.MAP v_analog -> 1\n'
        '.MAP b0 -> 3\n'
        '.MAP b1 -> 4\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG python\n'
        '.CODE\n'
        '# Convertimos el voltaje analógico (0-5V) a un entero de 4 bits (0-15)\n'
        'valor = int(round((v_analog / v_high) * 15.0))\n'
        'if valor > 15: valor = 15\n'
        'if valor < 0: valor = 0\n'
        '# Extraemos los dos bits menos significativos\n'
        'b0 = float((valor >> 0) & 1)\n'
        'b1 = float((valor >> 1) & 1)\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '.SUBCKT vhdl_shift\n'
        '.PORTS IN_DIG: a0 a1\n'
        '.PORTS OUT: y0 y1 y2 y3\n'
        '.MAP a0 -> 3\n'
        '.MAP a1 -> 4\n'
        '.MAP y0 -> 5\n'
        '.MAP y1 -> 6\n'
        '.MAP y2 -> 7\n'
        '.MAP y3 -> 8\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        '-- Multiplicar por 2 en binario es desplazar los bits a la izquierda\n'
        'y0 <= 0;\n'  # El bit 0 siempre es 0
        'y1 <= a0;\n' # El bit 1 es el antiguo bit 0
        'y2 <= a1;\n' # El bit 2 es el antiguo bit 1
        'y3 <= 0;\n'  # El bit 3 siempre es 0 (para este rango)
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '* DAC con Sumador Inversor y Inversor Final\n'
        'R_0 5 9 16k\n'
        'R_1 6 9 8k\n'
        'R_2 7 9 4k\n'
        'R_3 8 9 2k\n'
        'R_f 9 10 1k\n'
        '\n'
        'E_op 10 0 0 9 1e6\n'
        '\n'
        'R_inv1 10 11 1k\n'
        'R_inv2 11 12 1k\n'
        'E_inv 12 0 0 11 1e6\n'
        '\n'
        '.plot V(1) V(12)\n' # Entrada vs Salida DAC
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                fig, ax = plt.subplots(figsize=(10, 5))

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                for n in nodos_a_graficar:
                    if n in historial and n != 0:
                        ax.plot(tiempos, historial[n], label=f'V({n})')

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"
                ax.set_title(titulo)

                ax.set_xlabel('Time [s]')
                ax.set_ylabel('Voltage [V]')
                ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Sistema Mixto: Entrada (0-1V) -> ADC (Python) -> VHDL (Shift Left) -> DAC (Spice)\nV_in 1 0 …

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 4.A: Base estable con Vectores y Co-simulacion
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    c_code = f"""
    #include <math.h>
    void ejecutar_c(double *entradas, double *salidas) {{
    """
    for i, p in enumerate(puertos_in):
        c_code += f"    double {p} = entradas[{i}];\n"
    for p in puertos_out:
        c_code += f"    double {p} = 0.0;\n"

    c_code += codigo + "\n"

    for i, p in enumerate(puertos_out):
        c_code += f"    salidas[{i}] = {p};\n"
    c_code += "}\n"

    c_path = f"/content/{nombre}.c"
    so_path = f"/content/{nombre}.so"
    with open(c_path, 'w') as f:
        f.write(c_code)

    subprocess.run(['gcc', '-shared', '-fPIC', '-o', so_path, c_path, '-lm'], check=True)

    lib = ctypes.CDLL(so_path)
    lib.ejecutar_c.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
    return lib.ejecutar_c

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl:
            self.fuentes_pwl[nombre] = pwl

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        func_compilada = None
        if lang.lower() == 'c':
            func_compilada = compilar_c_y_cargar(nombre, codigo, dig_vars + analog_vars, out_vars)

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'func': func_compilada,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in out_vars}
        })

    def _estampar_bloques(self, A, b, idx, modo='dc'):
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

            g_out = 1.0 / bloque['r_out']
            for p in bloque['out_vars']:
                if p not in bloque['estado_anterior']: continue
                nodo_out = bloque['out_nodos'][bloque['out_vars'].index(p)]
                i_node = idx.get(nodo_out)
                if i_node is not None:
                    A[i_node, i_node] += g_out
                    estado = bloque['estado_anterior'][p]
                    v_pin = bloque['v_high'] if estado > 0.5 else bloque['v_low']
                    b[i_node] += v_pin * g_out

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        self._estampar_bloques(A, b, idx, modo)

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                local_env = {}
                for k, v in entradas.items(): local_env[k] = v
                for p in bloque['out_vars']: local_env[p] = 0.0

                for k, v in bloque['params'].items(): local_env[k.lower()] = v
                local_env['v_high'] = bloque['v_high']
                local_env['v_low'] = bloque['v_low']
                local_env['r_out'] = bloque['r_out']
                local_env['r_in'] = bloque['r_in']
                local_env['v_th'] = bloque['v_th']

                codigo_python = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', bloque['codigo'])

                try:
                    exec(codigo_python, {}, local_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(local_env[p])
                except Exception as e:
                    print(f"Error en bloque Python {bloque['nombre']}: {e}")

            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*[entradas[k] for k in bloque['dig_vars'] + bloque['analog_vars']])
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

            elif bloque['lang'] == 'vhdl':
                res = {}
                for p in bloque['out_vars']: res[p] = 0.0

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                sentencias = [s.strip() for s in codigo_limpio.split(';') if s.strip()]

                for sent in sentencias:
                    sent = sent.replace("'", "")
                    sent = re.sub(r'\bAND\b', ' and ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bOR\b', ' or ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bNOT\b', ' not ', sent, flags=re.IGNORECASE)
                    sent = re.sub(r'\bXOR\b', ' ^ ', sent, flags=re.IGNORECASE)

                    sent = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', sent)

                    match_when = re.match(r'(\w+)\s*<=\s*(.*?)\s+WHEN\s+(.*?)\s+ELSE\s+(.*)', sent, re.IGNORECASE)
                    if match_when:
                        out_var = match_when.group(1).strip()
                        val_true = match_when.group(2).strip()
                        cond = match_when.group(3).strip()
                        val_false = match_when.group(4).strip()

                        local_env = dict(entradas)
                        try:
                            cond_eval = cond.replace(' = ', ' == ')
                            v_true = float(eval(val_true, {}, local_env))
                            v_false = float(eval(val_false, {}, local_env))
                            if eval(cond_eval, {}, local_env):
                                res[out_var] = v_true
                            else:
                                res[out_var] = v_false
                        except:
                            pass
                    else:
                        match_simple = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match_simple:
                            out_var = match_simple.group(1).strip()
                            expr = match_simple.group(2).strip()
                            local_env = dict(entradas)
                            try:
                                res[out_var] = float(eval(expr, {}, local_env))
                            except:
                                pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res[p]

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': []}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                    if match:
                        try:
                            nodo = int(match.group(1))
                            directivas['plot'].append(nodo)
                        except ValueError:
                            pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and 'PWL' in linea.upper():
                    match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pwl:
                        raise ValueError("Formato PWL invalido (faltan parentesis)")
                    contenido = match_pwl.group(1).strip()
                    pwl_tokens = contenido.replace(',', ' ').split()
                    pwl_puntos = []
                    for i in range(0, len(pwl_tokens), 2):
                        if i+1 < len(pwl_tokens):
                            t_val = sufijo_a_valor(pwl_tokens[i])
                            v_val = sufijo_a_valor(pwl_tokens[i+1])
                            pwl_puntos.append((t_val, v_val))
                    if not pwl_puntos:
                        raise ValueError("PWL sin puntos")
                    c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Sistema Mixto con Vectores: Entrada (0-1V) -> ADC -> VHDL (Shift Left) -> DAC\n'
        'V_in 1 0 PWL(0 0 10m 1 20m 0)\n'
        '\n'
        '.SUBCKT adc_python\n'
        '.PORTS IN_ANALOG: v_analog\n'
        '.PORTS OUT: B[0..1]\n'
        '.MAP v_analog -> 1\n'
        '.MAP B -> 3..4\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG python\n'
        '.CODE\n'
        'valor = int(round((v_analog / v_high) * 15.0))\n'
        'if valor > 15: valor = 15\n'
        'if valor < 0: valor = 0\n'
        'B[0] = float((valor >> 0) & 1)\n'
        'B[1] = float((valor >> 1) & 1)\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '.SUBCKT vhdl_shift\n'
        '.PORTS IN_DIG: A[0..1]\n'
        '.PORTS OUT: Y[0..3]\n'
        '.MAP A -> 3..4\n'
        '.MAP Y -> 5..8\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Y[0] <= 0;\n'
        'Y[1] <= A[0];\n'
        'Y[2] <= A[1];\n'
        'Y[3] <= 0;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        'R_0 5 9 16k\n'
        'R_1 6 9 8k\n'
        'R_2 7 9 4k\n'
        'R_3 8 9 2k\n'
        'R_f 9 10 1k\n'
        '\n'
        'E_op 10 0 0 9 1e6\n'
        '\n'
        'R_inv1 10 11 1k\n'
        'R_inv2 11 12 1k\n'
        'E_inv 12 0 0 11 1e6\n'
        '\n'
        '.plot V(1) V(12)\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                fig, ax = plt.subplots(figsize=(10, 5))

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                for n in nodos_a_graficar:
                    if n in historial and n != 0:
                        ax.plot(tiempos, historial[n], label=f'V({n})')

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"
                ax.set_title(titulo)

                ax.set_xlabel('Time [s]')
                ax.set_ylabel('Voltage [V]')
                ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Sistema Mixto con Vectores: Entrada (0-1V) -> ADC -> VHDL (Shift Left) -> DAC\nV_in 1 0 PWL(…

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 4.T: Corrección total del estampado MNA para VHDL
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    return None

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.fuentes_pulse = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None, pulse=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl: self.fuentes_pwl[nombre] = pwl
        if pulse: self.fuentes_pulse[nombre] = pulse

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        if nombre in self.fuentes_pulse:
            v1, v2, delay, rise, fall, width, period = self.fuentes_pulse[nombre]
            if tiempo < delay: return v1
            t_rel = (tiempo - delay) % period
            if t_rel < rise: return v1 + (v2 - v1) * (t_rel / rise)
            if t_rel < rise + width: return v2
            if t_rel < rise + width + fall: return v2 + (v1 - v2) * ((t_rel - rise - width) / fall)
            return v1
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        # Registrar los pines de salida como fuentes de voltaje para MNA
        for i, p in enumerate(out_vars):
            nombre_fuente = f"V_{nombre}_{p}"
            self.fuentes_v.append(nombre_fuente)
            self.componentes.append(('V_BLOQUE', nombre_fuente, nodos_out[i], 0, 0.0))

        estado_registros = {p: params.get('V_LOW', 0.0) for p in out_vars}
        estado_registros['_prev_clk'] = 0.0

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': estado_registros
        })

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)

        # Fuentes de voltaje totales
        todas_fuentes_v = list(self.fuentes_v)
        n_fv = len(todas_fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(todas_fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'V_BLOQUE':
                _, nombre, nodo_out, _, _ = comp
                k = idx_fv[nombre]
                i1 = idx.get(nodo_out)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                b[k] = 0.0 # Se actualiza antes de resolver
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        # Estampar R_IN de los bloques
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

        # Actualizar los valores de las fuentes de bloque
        for bloque in self.bloques_codigo:
            for i, p in enumerate(bloque['out_vars']):
                nombre_fuente = f"V_{bloque['nombre']}_{p}"
                k = idx_fv[nombre_fuente]
                estado = bloque['estado_anterior'][p]
                b[k] = bloque['v_high'] if estado > 0.5 else bloque['v_low']

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales, tiempo_actual, dt):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'vhdl':
                res = dict(bloque['estado_anterior'])

                clk_actual = entradas.get('clk', 0.0)
                clk_previo = bloque['estado_anterior'].get('_prev_clk', 0.0)
                flanco_subida = (clk_actual > 0.5) and (clk_previo < 0.5)
                res['_prev_clk'] = clk_actual

                local_env = dict(entradas)
                for p in bloque['out_vars']:
                    local_env[p] = bloque['estado_anterior'].get(p, 0.0)
                local_env = {re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', k): v for k, v in local_env.items()}

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])

                lineas = [l.strip() for l in codigo_limpio.splitlines() if l.strip()]
                en_process = False

                for linea in lineas:
                    if re.match(r'process\s*\(', linea, re.IGNORECASE):
                        en_process = True
                        continue
                    if re.match(r'end\s+process', linea, re.IGNORECASE):
                        en_process = False
                        continue
                    if re.match(r'if\s+rising_edge', linea, re.IGNORECASE):
                        continue
                    if re.match(r'end\s+if', linea, re.IGNORECASE):
                        continue

                    match = re.match(r'(\w+)\s*<=\s*(.*)', linea, re.IGNORECASE)
                    if match:
                        out_var = match.group(1).strip()
                        expr = match.group(2).strip().rstrip(';')

                        if en_process and not flanco_subida:
                            continue

                        sent = expr.replace("'", "")
                        sent = re.sub(r'\bAND\b', ' and ', sent, flags=re.IGNORECASE)
                        sent = re.sub(r'\bOR\b', ' or ', sent, flags=re.IGNORECASE)
                        sent = re.sub(r'\bNOT\b', ' not ', sent, flags=re.IGNORECASE)
                        sent = re.sub(r'\bXOR\b', ' ^ ', sent, flags=re.IGNORECASE)
                        try:
                            res[out_var] = float(eval(sent, {}, local_env))
                        except:
                            pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res.get(p, 0.0)

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes, 0.0, 1e-6)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]

            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            self._ejecutar_logica_bloques(voltajes, t_actual, dt)

            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': []}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                    if match:
                        try:
                            nodo = int(match.group(1))
                            directivas['plot'].append(nodo)
                        except ValueError:
                            pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and 'PWL' in linea.upper():
                    match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pwl:
                        raise ValueError("Formato PWL invalido")
                    contenido = match_pwl.group(1).strip()
                    pwl_tokens = contenido.replace(',', ' ').split()
                    pwl_puntos = []
                    for i in range(0, len(pwl_tokens), 2):
                        if i+1 < len(pwl_tokens):
                            t_val = sufijo_a_valor(pwl_tokens[i])
                            v_val = sufijo_a_valor(pwl_tokens[i+1])
                            pwl_puntos.append((t_val, v_val))
                    c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                elif tipo == 'V' and 'PULSE' in linea.upper():
                    match_pulse = re.search(r'PULSE\s*\((.*?)\)', linea, re.IGNORECASE)
                    if not match_pulse:
                        raise ValueError("Formato PULSE invalido")
                    contenido = match_pulse.group(1).strip()
                    pulse_tokens = contenido.replace(',', ' ').split()
                    vals = [sufijo_a_valor(t) for t in pulse_tokens]
                    while len(vals) < 7: vals.append(0.0)
                    c.V(nombre, n1, n2, 0.0, pulse=tuple(vals[:7]))
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Prueba VHDL: Entrada directa 101 y DAC con Opamp\n'
        'V_b0 3 0 5\n'
        'V_b1 4 0 0\n'
        'V_b2 5 0 5\n'
        'V_clk 2 0 PULSE(0 5 0 1u 1u 1m 2m)\n'
        '\n'
        '.SUBCKT registro_vhdl\n'
        '.PORTS IN_DIG: clk\n'
        '.PORTS IN_DIG: data_in[0..2]\n'
        '.PORTS OUT: Q_act[0..2]\n'
        '.PORTS OUT: Q_ret[0..2]\n'
        '.MAP clk -> 2\n'
        '.MAP data_in -> 3..5\n'
        '.MAP Q_act -> 7..9\n'
        '.MAP Q_ret -> 11..13\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_IN=1meg V_TH=2.5\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Q_act[0] <= data_in[0];\n'
        'Q_act[1] <= data_in[1];\n'
        'Q_act[2] <= data_in[2];\n'
        'process(clk) begin\n'
        '  if rising_edge(clk) then\n'
        '    Q_ret[0] <= data_in[0];\n'
        '    Q_ret[1] <= data_in[1];\n'
        '    Q_ret[2] <= data_in[2];\n'
        '  end if;\n'
        'end process;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '* DAC 1 (Actual)\n'
        'R_a0 7 15 10k\n'
        'R_a1 8 15 20k\n'
        'R_a2 9 15 40k\n'
        'R_fa 15 17 10k\n'
        'E_op1 17 0 0 15 1e6\n'
        'R_ia1 17 18 1k\n'
        'R_ia2 18 19 1k\n'
        'E_in1 19 0 0 18 1e6\n'
        '\n'
        '.plot V(19)\n'
        '.tran 10u 10m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                fig, ax = plt.subplots(figsize=(10, 5))

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                for n in nodos_a_graficar:
                    if n in historial and n != 0:
                        ax.plot(tiempos, historial[n], label=f'V({n})')

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"
                ax.set_title(titulo)

                ax.set_xlabel('Time [s]')
                ax.set_ylabel('Voltage [V]')
                ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Prueba VHDL: Entrada directa 101 y DAC con Opamp\nV_b0 3 0 5\nV_b1 4 0 0\nV_b2 5 0 5\nV_clk …

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 4.F: Soporte para graficas con subplots (SUBPLOT=ON)
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    return None

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.fuentes_pulse = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None, pulse=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl: self.fuentes_pwl[nombre] = pwl
        if pulse: self.fuentes_pulse[nombre] = pulse

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        if nombre in self.fuentes_pulse:
            v1, v2, delay, rise, fall, width, period = self.fuentes_pulse[nombre]
            if tiempo < delay: return v1
            t_rel = (tiempo - delay) % period
            if t_rel < rise: return v1 + (v2 - v1) * (t_rel / rise)
            if t_rel < rise + width: return v2
            if t_rel < rise + width + fall: return v2 + (v1 - v2) * ((t_rel - rise - width) / fall)
            return v1
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        for i, p in enumerate(out_vars):
            nombre_fuente = f"V_{nombre}_{p}"
            self.fuentes_v.append(nombre_fuente)
            self.componentes.append(('V_BLOQUE', nombre_fuente, nodos_out[i], 0, 0.0))

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in out_vars},
            '_prev_clk': 0.0
        })

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'V_BLOQUE':
                _, nombre, nodo_out, _, _ = comp
                k = idx_fv[nombre]
                i1 = idx.get(nodo_out)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                b[k] = 0.0
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

        for bloque in self.bloques_codigo:
            for i, p in enumerate(bloque['out_vars']):
                nombre_fuente = f"V_{bloque['nombre']}_{p}"
                k = idx_fv[nombre_fuente]
                estado = bloque['estado_anterior'][p]
                b[k] = bloque['v_high'] if estado > 0.5 else bloque['v_low']

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'vhdl':
                res = dict(bloque['estado_anterior'])

                clk_actual = entradas.get('clk', 0.0)
                clk_previo = bloque['_prev_clk']
                flanco_subida = (clk_actual > 0.5) and (clk_previo < 0.5)
                bloque['_prev_clk'] = clk_actual

                local_env = dict(entradas)
                for p in bloque['out_vars']:
                    local_env[p] = bloque['estado_anterior'].get(p, 0.0)

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])

                match_process = re.search(r'process\s*\(.*?\)\s*begin(.*?)end\s*process', codigo_limpio, re.IGNORECASE | re.DOTALL)

                if match_process:
                    cuerpo_process = match_process.group(1)
                    cuerpo_process = re.sub(r'if\s+rising_edge\s*\(.*?\)\s*then', '', cuerpo_process, flags=re.IGNORECASE)
                    cuerpo_process = re.sub(r'end\s+if', '', cuerpo_process, flags=re.IGNORECASE)

                    codigo_fuera = codigo_limpio.replace(match_process.group(0), '')

                    for sent in [s.strip() for s in codigo_fuera.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                    if flanco_subida:
                        for sent in [s.strip() for s in cuerpo_process.split(';') if s.strip()]:
                            match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                            if match:
                                out_var = match.group(1).strip()
                                expr = match.group(2).strip().rstrip(';')
                                sent_eval = expr.replace("'", "")
                                sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                                try: res[out_var] = float(eval(sent_eval, {}, local_env))
                                except: pass
                else:
                    for sent in [s.strip() for s in codigo_limpio.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res.get(p, 0.0)

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': [], 'subplot': False}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                # Analizar parametros del .plot
                for t in tokens[1:]:
                    if 'SUBPLOT=ON' in t.upper():
                        directivas['subplot'] = True
                    elif 'SUBPLOT=OFF' in t.upper():
                        directivas['subplot'] = False
                    else:
                        match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                        if match:
                            try:
                                nodo = int(match.group(1))
                                directivas['plot'].append(nodo)
                            except ValueError:
                                pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and len(tokens) > 3:
                    token_valor = tokens[3].upper()
                    if token_valor.startswith('PWL('):
                        match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pwl:
                            contenido = match_pwl.group(1).strip()
                            pwl_tokens = contenido.replace(',', ' ').split()
                            pwl_puntos = []
                            for i in range(0, len(pwl_tokens), 2):
                                if i+1 < len(pwl_tokens):
                                    t_val = sufijo_a_valor(pwl_tokens[i])
                                    v_val = sufijo_a_valor(pwl_tokens[i+1])
                                    pwl_puntos.append((t_val, v_val))
                            c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                        else:
                            raise ValueError("Formato PWL invalido")
                    elif token_valor.startswith('PULSE('):
                        match_pulse = re.search(r'PULSE\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pulse:
                            contenido = match_pulse.group(1).strip()
                            pulse_tokens = contenido.replace(',', ' ').split()
                            vals = [sufijo_a_valor(t) for t in pulse_tokens]
                            while len(vals) < 7: vals.append(0.0)
                            c.V(nombre, n1, n2, 0.0, pulse=tuple(vals[:7]))
                        else:
                            raise ValueError("Formato PULSE invalido")
                    else:
                        valor = sufijo_a_valor(tokens[3])
                        getattr(c, tipo)(nombre, n1, n2, valor)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Prueba de Dos DACs y Reloj Invertido\n'
        'V_b0 3 0 5\n'
        'V_b1 4 0 0\n'
        'V_b2 5 0 5\n'
        'V_clk 2 0 PULSE(5 0 0 1u 1u 1m 2m)\n'
        '\n'
        '.SUBCKT registro_vhdl\n'
        '.PORTS IN_DIG: clk\n'
        '.PORTS IN_DIG: data_in_0 data_in_1 data_in_2\n'
        '.PORTS OUT: Q_act_0 Q_act_1 Q_act_2\n'
        '.PORTS OUT: Q_ret_0 Q_ret_1 Q_ret_2\n'
        '.MAP clk -> 2\n'
        '.MAP data_in_0 -> 3\n'
        '.MAP data_in_1 -> 4\n'
        '.MAP data_in_2 -> 5\n'
        '.MAP Q_act_0 -> 7\n'
        '.MAP Q_act_1 -> 8\n'
        '.MAP Q_act_2 -> 9\n'
        '.MAP Q_ret_0 -> 11\n'
        '.MAP Q_ret_1 -> 12\n'
        '.MAP Q_ret_2 -> 13\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Q_act_0 <= data_in_0;\n'
        'Q_act_1 <= data_in_1;\n'
        'Q_act_2 <= data_in_2;\n'
        'process(clk) begin\n'
        '  if rising_edge(clk) then\n'
        '    Q_ret_0 <= data_in_0;\n'
        '    Q_ret_1 <= data_in_1;\n'
        '    Q_ret_2 <= data_in_2;\n'
        '  end if;\n'
        'end process;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '* DAC 1 (Entrada - Combinacional)\n'
        'R_a0 7 15 8k\n'
        'R_a1 8 15 4k\n'
        'R_a2 9 15 2k\n'
        'R_fa 15 17 1k\n'
        'E_op1 17 0 0 15 1e6\n'
        'R_ia1 17 18 1k\n'
        'R_ia2 18 19 1k\n'
        'E_in1 19 0 0 18 1e6\n'
        '\n'
        '* DAC 2 (Salida - Retrasada)\n'
        'R_b0 11 20 8k\n'
        'R_b1 12 20 4k\n'
        'R_b2 13 20 2k\n'
        'R_fb 20 22 1k\n'
        'E_op2 22 0 0 20 1e6\n'
        'R_ib1 22 23 1k\n'
        'R_ib2 23 24 1k\n'
        'E_in2 24 0 0 23 1e6\n'
        '\n'
        '.plot V(2) V(19) V(24) SUBPLOT=ON\n'
        '.tran 10u 10m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"

                if directivas['subplot']:
                    # Crear subplots verticales
                    fig, axes = plt.subplots(nrows=len(nodos_a_graficar), ncols=1, figsize=(10, 3*len(nodos_a_graficar)), sharex=True)
                    if len(nodos_a_graficar) == 1: axes = [axes] # Asegurar iterable

                    for ax, n in zip(axes, nodos_a_graficar):
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')
                            ax.set_ylabel('Voltaje [V]')
                            ax.grid(True)
                            ax.legend(loc='upper left')

                    axes[-1].set_xlabel('Time [s]')
                    fig.suptitle(titulo)
                else:
                    # Grafica unica superpuesta
                    fig, ax = plt.subplots(figsize=(10, 5))
                    for n in nodos_a_graficar:
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')

                    ax.set_title(titulo)
                    ax.set_xlabel('Time [s]')
                    ax.set_ylabel('Voltage [V]')
                    ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Prueba de Dos DACs y Reloj Invertido\nV_b0 3 0 5\nV_b1 4 0 0\nV_b2 5 0 5\nV_clk 2 0 PULSE(5 …

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 4.J: Editor con fuente mas grande (CSS)
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files as colab_files

# ---------------------------------------------------------
# 0. ESTILO CSS PARA AGRANDAR EL EDITOR
# ---------------------------------------------------------
display(HTML("""
<style>
    /* Aumentar el tamaño del texto dentro del editor de netlist */
    .widget-textarea textarea {
        font-size: 16px !important;
        font-family: 'Courier New', monospace !important;
        line-height: 1.4 !important;
    }
    /* Aumentar el tamano de la fuente de los botones */
    .widget-button {
        font-size: 14px !important;
        font-weight: bold !important;
    }
</style>
"""))

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    return None

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.fuentes_pulse = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None, pulse=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl: self.fuentes_pwl[nombre] = pwl
        if pulse: self.fuentes_pulse[nombre] = pulse

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        if nombre in self.fuentes_pulse:
            v1, v2, delay, rise, fall, width, period = self.fuentes_pulse[nombre]
            if tiempo < delay: return v1
            t_rel = (tiempo - delay) % period
            if t_rel < rise: return v1 + (v2 - v1) * (t_rel / rise)
            if t_rel < rise + width: return v2
            if t_rel < rise + width + fall: return v2 + (v1 - v2) * ((t_rel - rise - width) / fall)
            return v1
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        for i, p in enumerate(out_vars):
            nombre_fuente = f"V_{nombre}_{p}"
            self.fuentes_v.append(nombre_fuente)
            self.componentes.append(('V_BLOQUE', nombre_fuente, nodos_out[i], 0, 0.0))

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in out_vars},
            '_prev_clk': 0.0
        })

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'V_BLOQUE':
                _, nombre, nodo_out, _, _ = comp
                k = idx_fv[nombre]
                i1 = idx.get(nodo_out)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                b[k] = 0.0
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

        for bloque in self.bloques_codigo:
            for i, p in enumerate(bloque['out_vars']):
                nombre_fuente = f"V_{bloque['nombre']}_{p}"
                k = idx_fv[nombre_fuente]
                estado = bloque['estado_anterior'][p]
                b[k] = bloque['v_high'] if estado > 0.5 else bloque['v_low']

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                global_env = dict(entradas)
                for p in bloque['out_vars']:
                    global_env[p] = bloque['estado_anterior'].get(p, 0.0)

                for k, v in bloque['params'].items(): global_env[k.lower()] = v
                global_env['v_high'] = bloque['v_high']
                global_env['v_low'] = bloque['v_low']

                codigo_python = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', bloque['codigo'])

                try:
                    exec(codigo_python, global_env, global_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(global_env[p])
                except Exception as e:
                    pass

            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*[entradas[k] for k in bloque['dig_vars'] + bloque['analog_vars']])
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

            elif bloque['lang'] == 'vhdl':
                res = dict(bloque['estado_anterior'])

                clk_actual = entradas.get('clk', 0.0)
                clk_previo = bloque['_prev_clk']
                flanco_subida = (clk_actual > 0.5) and (clk_previo < 0.5)
                bloque['_prev_clk'] = clk_actual

                local_env = dict(entradas)
                for p in bloque['out_vars']:
                    local_env[p] = bloque['estado_anterior'].get(p, 0.0)

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                codigo_limpio = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', codigo_limpio)

                match_process = re.search(r'process\s*\(.*?\)\s*begin(.*?)end\s*process', codigo_limpio, re.IGNORECASE | re.DOTALL)

                if match_process:
                    cuerpo_process = match_process.group(1)
                    cuerpo_process = re.sub(r'if\s+rising_edge\s*\(.*?\)\s*then', '', cuerpo_process, flags=re.IGNORECASE)
                    cuerpo_process = re.sub(r'end\s+if', '', cuerpo_process, flags=re.IGNORECASE)

                    codigo_fuera = codigo_limpio.replace(match_process.group(0), '')

                    for sent in [s.strip() for s in codigo_fuera.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                    if flanco_subida:
                        for sent in [s.strip() for s in cuerpo_process.split(';') if s.strip()]:
                            match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                            if match:
                                out_var = match.group(1).strip()
                                expr = match.group(2).strip().rstrip(';')
                                sent_eval = expr.replace("'", "")
                                sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                                try: res[out_var] = float(eval(sent_eval, {}, local_env))
                                except: pass
                else:
                    for sent in [s.strip() for s in codigo_limpio.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res.get(p, 0.0)

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': [], 'subplot': False}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    if 'SUBPLOT=ON' in t.upper():
                        directivas['subplot'] = True
                    elif 'SUBPLOT=OFF' in t.upper():
                        directivas['subplot'] = False
                    else:
                        match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                        if match:
                            try:
                                nodo = int(match.group(1))
                                directivas['plot'].append(nodo)
                            except ValueError:
                                pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and len(tokens) > 3:
                    token_valor = tokens[3].upper()
                    if token_valor.startswith('PWL('):
                        match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pwl:
                            contenido = match_pwl.group(1).strip()
                            pwl_tokens = contenido.replace(',', ' ').split()
                            pwl_puntos = []
                            for i in range(0, len(pwl_tokens), 2):
                                if i+1 < len(pwl_tokens):
                                    t_val = sufijo_a_valor(pwl_tokens[i])
                                    v_val = sufijo_a_valor(pwl_tokens[i+1])
                                    pwl_puntos.append((t_val, v_val))
                            c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                        else:
                            raise ValueError("Formato PWL invalido")
                    elif token_valor.startswith('PULSE('):
                        match_pulse = re.search(r'PULSE\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pulse:
                            contenido = match_pulse.group(1).strip()
                            pulse_tokens = contenido.replace(',', ' ').split()
                            vals = [sufijo_a_valor(t) for t in pulse_tokens]
                            while len(vals) < 7: vals.append(0.0)
                            c.V(nombre, n1, n2, 0.0, pulse=tuple(vals[:7]))
                        else:
                            raise ValueError("Formato PULSE invalido")
                    else:
                        valor = sufijo_a_valor(tokens[3])
                        getattr(c, tipo)(nombre, n1, n2, valor)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None}

editor = widgets.Textarea(
    value=(
        '* Sistema Completo: ADC (Python) -> VHDL -> 2 DACs + Reloj\n'
        'V_in 1 0 PWL(0 0 10m 5 20m 0)\n'
        'V_clk 2 0 PULSE(0 5 1m 1u 1u 1m 2m)\n'
        '\n'
        '.SUBCKT adc_python\n'
        '.PORTS IN_ANALOG: v_analog\n'
        '.PORTS OUT: adc_bits[0..2]\n'
        '.MAP v_analog -> 1\n'
        '.MAP adc_bits -> 3..5\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG python\n'
        '.CODE\n'
        'valor = int(round((v_analog / v_high) * 7.0))\n'
        'if valor > 7: valor = 7\n'
        'if valor < 0: valor = 0\n'
        'adc_bits[0] = float((valor >> 0) & 1)\n'
        'adc_bits[1] = float((valor >> 1) & 1)\n'
        'adc_bits[2] = float((valor >> 2) & 1)\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '.SUBCKT registro_vhdl\n'
        '.PORTS IN_DIG: clk\n'
        '.PORTS IN_DIG: data_in[0..2]\n'
        '.PORTS OUT: Q_act[0..2]\n'
        '.PORTS OUT: Q_ret[0..2]\n'
        '.MAP clk -> 2\n'
        '.MAP data_in -> 3..5\n'
        '.MAP Q_act -> 7..9\n'
        '.MAP Q_ret -> 11..13\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Q_act[0] <= data_in[0];\n'
        'Q_act[1] <= data_in[1];\n'
        'Q_act[2] <= data_in[2];\n'
        'process(clk) begin\n'
        '  if rising_edge(clk) then\n'
        '    Q_ret[0] <= data_in[0];\n'
        '    Q_ret[1] <= data_in[1];\n'
        '    Q_ret[2] <= data_in[2];\n'
        '  end if;\n'
        'end process;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '* DAC 1 (Entrada - Combinacional)\n'
        'R_a0 7 15 8k\n'
        'R_a1 8 15 4k\n'
        'R_a2 9 15 2k\n'
        'R_fa 15 17 1k\n'
        'E_op1 17 0 0 15 1e6\n'
        'R_ia1 17 18 1k\n'
        'R_ia2 18 19 1k\n'
        'E_in1 19 0 0 18 1e6\n'
        '\n'
        '* DAC 2 (Salida - Retrasada)\n'
        'R_b0 11 20 8k\n'
        'R_b1 12 20 4k\n'
        'R_b2 13 20 2k\n'
        'R_fb 20 22 1k\n'
        'E_op2 22 0 0 20 1e6\n'
        'R_ib1 22 23 1k\n'
        'R_ib2 23 24 1k\n'
        'E_in2 24 0 0 23 1e6\n'
        '\n'
        '.plot V(1) V(2) V(19) V(24) SUBPLOT=ON\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_simular(b):
    with output_area:
        clear_output()
        try:
            c, directivas, errores = parsear_netlist(editor.value)
            if errores:
                print("Advertencias:")
                for e in errores: print(f"  - {e}")
                print()
            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"

                if directivas['subplot']:
                    fig, axes = plt.subplots(nrows=len(nodos_a_graficar), ncols=1, figsize=(10, 3*len(nodos_a_graficar)), sharex=True)
                    if len(nodos_a_graficar) == 1: axes = [axes]

                    for ax, n in zip(axes, nodos_a_graficar):
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')
                            ax.set_ylabel('Voltaje [V]')
                            ax.grid(True)
                            ax.legend(loc='upper left')

                    axes[-1].set_xlabel('Time [s]')
                    fig.suptitle(titulo)
                else:
                    fig, ax = plt.subplots(figsize=(10, 5))
                    for n in nodos_a_graficar:
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')

                    ax.set_title(titulo)
                    ax.set_xlabel('Time [s]')
                    ax.set_ylabel('Voltage [V]')
                    ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            print("ERROR: matriz singular.")
        except Exception as e:
            print(f"ERROR: {e}")

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_simular])
display(barra, uploader, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

Textarea(value='* Sistema Completo: ADC (Python) -> VHDL -> 2 DACs + Reloj\nV_in 1 0 PWL(0 0 10m 5 20m 0)\nV_c…

Output()

In [ ]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 5.B: Nivel 1 (sin boton de zoom)
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files as colab_files, drive

# ---------------------------------------------------------
# 0. ESTILO CSS
# ---------------------------------------------------------
display(HTML("""
<style>
    .widget-textarea textarea {
        font-size: 16px !important;
        font-family: 'Courier New', monospace !important;
        line-height: 1.4 !important;
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
        border: 1px solid #444 !important;
    }
    .widget-button {
        font-size: 14px !important;
        font-weight: bold !important;
    }
    .error-panel {
        background-color: #f8d7da;
        color: #721c24;
        padding: 10px;
        border-radius: 5px;
        border: 1px solid #f5c6cb;
        font-family: monospace;
        white-space: pre-wrap;
    }
    .success-panel {
        background-color: #d4edda;
        color: #155724;
        padding: 10px;
        border-radius: 5px;
        border: 1px solid #c3e6cb;
        font-family: monospace;
        white-space: pre-wrap;
    }
</style>
"""))

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    return None

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.fuentes_pulse = {}
        self.bloques_codigo = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None, pulse=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl: self.fuentes_pwl[nombre] = pwl
        if pulse: self.fuentes_pulse[nombre] = pulse

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        if nombre in self.fuentes_pulse:
            v1, v2, delay, rise, fall, width, period = self.fuentes_pulse[nombre]
            if tiempo < delay: return v1
            t_rel = (tiempo - delay) % period
            if t_rel < rise: return v1 + (v2 - v1) * (t_rel / rise)
            if t_rel < rise + width: return v2
            if t_rel < rise + width + fall: return v2 + (v1 - v2) * ((t_rel - rise - width) / fall)
            return v1
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        for i, p in enumerate(out_vars):
            nombre_fuente = f"V_{nombre}_{p}"
            self.fuentes_v.append(nombre_fuente)
            self.componentes.append(('V_BLOQUE', nombre_fuente, nodos_out[i], 0, 0.0))

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in out_vars},
            '_prev_clk': 0.0
        })

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'V_BLOQUE':
                _, nombre, nodo_out, _, _ = comp
                k = idx_fv[nombre]
                i1 = idx.get(nodo_out)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                b[k] = 0.0
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

        for bloque in self.bloques_codigo:
            for i, p in enumerate(bloque['out_vars']):
                nombre_fuente = f"V_{bloque['nombre']}_{p}"
                k = idx_fv[nombre_fuente]
                estado = bloque['estado_anterior'][p]
                b[k] = bloque['v_high'] if estado > 0.5 else bloque['v_low']

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                global_env = dict(entradas)
                for p in bloque['out_vars']:
                    global_env[p] = bloque['estado_anterior'].get(p, 0.0)

                for k, v in bloque['params'].items(): global_env[k.lower()] = v
                global_env['v_high'] = bloque['v_high']
                global_env['v_low'] = bloque['v_low']

                codigo_python = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', bloque['codigo'])

                try:
                    exec(codigo_python, global_env, global_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(global_env[p])
                except Exception as e:
                    pass

            elif bloque['lang'] == 'c':
                in_arr = (ctypes.c_double * len(entradas))(*[entradas[k] for k in bloque['dig_vars'] + bloque['analog_vars']])
                out_arr = (ctypes.c_double * len(bloque['out_vars']))()
                bloque['func'](in_arr, out_arr)
                for i, p in enumerate(bloque['out_vars']):
                    bloque['estado_anterior'][p] = out_arr[i]

            elif bloque['lang'] == 'vhdl':
                res = dict(bloque['estado_anterior'])

                clk_actual = entradas.get('clk', 0.0)
                clk_previo = bloque['_prev_clk']
                flanco_subida = (clk_actual > 0.5) and (clk_previo < 0.5)
                bloque['_prev_clk'] = clk_actual

                local_env = dict(entradas)
                for p in bloque['out_vars']:
                    local_env[p] = bloque['estado_anterior'].get(p, 0.0)

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                codigo_limpio = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', codigo_limpio)

                match_process = re.search(r'process\s*\(.*?\)\s*begin(.*?)end\s*process', codigo_limpio, re.IGNORECASE | re.DOTALL)

                if match_process:
                    cuerpo_process = match_process.group(1)
                    cuerpo_process = re.sub(r'if\s+rising_edge\s*\(.*?\)\s*then', '', cuerpo_process, flags=re.IGNORECASE)
                    cuerpo_process = re.sub(r'end\s+if', '', cuerpo_process, flags=re.IGNORECASE)

                    codigo_fuera = codigo_limpio.replace(match_process.group(0), '')

                    for sent in [s.strip() for s in codigo_fuera.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                    if flanco_subida:
                        for sent in [s.strip() for s in cuerpo_process.split(';') if s.strip()]:
                            match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                            if match:
                                out_var = match.group(1).strip()
                                expr = match.group(2).strip().rstrip(';')
                                sent_eval = expr.replace("'", "")
                                sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                                try: res[out_var] = float(eval(sent_eval, {}, local_env))
                                except: pass
                else:
                    for sent in [s.strip() for s in codigo_limpio.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res.get(p, 0.0)

    def punto_operacion(self):
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': [], 'subplot': False}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.ends':
                c.agregar_bloque_codigo(
                    subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                    subckt_lang, "\n".join(subckt_codigo), subckt_params
                )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    if 'SUBPLOT=ON' in t.upper():
                        directivas['subplot'] = True
                    elif 'SUBPLOT=OFF' in t.upper():
                        directivas['subplot'] = False
                    else:
                        match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                        if match:
                            try:
                                nodo = int(match.group(1))
                                directivas['plot'].append(nodo)
                            except ValueError:
                                pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and len(tokens) > 3:
                    token_valor = tokens[3].upper()
                    if token_valor.startswith('PWL('):
                        match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pwl:
                            contenido = match_pwl.group(1).strip()
                            pwl_tokens = contenido.replace(',', ' ').split()
                            pwl_puntos = []
                            for i in range(0, len(pwl_tokens), 2):
                                if i+1 < len(pwl_tokens):
                                    t_val = sufijo_a_valor(pwl_tokens[i])
                                    v_val = sufijo_a_valor(pwl_tokens[i+1])
                                    pwl_puntos.append((t_val, v_val))
                            c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                        else:
                            raise ValueError("Formato PWL invalido")
                    elif token_valor.startswith('PULSE('):
                        match_pulse = re.search(r'PULSE\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pulse:
                            contenido = match_pulse.group(1).strip()
                            pulse_tokens = contenido.replace(',', ' ').split()
                            vals = [sufijo_a_valor(t) for t in pulse_tokens]
                            while len(vals) < 7: vals.append(0.0)
                            c.V(nombre, n1, n2, 0.0, pulse=tuple(vals[:7]))
                        else:
                            raise ValueError("Formato PULSE invalido")
                    else:
                        valor = sufijo_a_valor(tokens[3])
                        getattr(c, tipo)(nombre, n1, n2, valor)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None, "drive_mounted": False}

editor = widgets.Textarea(
    value=(
        '* Sistema Completo: ADC (Python) -> VHDL -> 2 DACs + Reloj\n'
        'V_in 1 0 PWL(0 0 10m 5 20m 0)\n'
        'V_clk 2 0 PULSE(0 5 1m 1u 1u 1m 2m)\n'
        '\n'
        '.SUBCKT adc_python\n'
        '.PORTS IN_ANALOG: v_analog\n'
        '.PORTS OUT: adc_bits[0..2]\n'
        '.MAP v_analog -> 1\n'
        '.MAP adc_bits -> 3..5\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG python\n'
        '.CODE\n'
        'valor = int(round((v_analog / v_high) * 7.0))\n'
        'if valor > 7: valor = 7\n'
        'if valor < 0: valor = 0\n'
        'adc_bits[0] = float((valor >> 0) & 1)\n'
        'adc_bits[1] = float((valor >> 1) & 1)\n'
        'adc_bits[2] = float((valor >> 2) & 1)\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '.SUBCKT registro_vhdl\n'
        '.PORTS IN_DIG: clk\n'
        '.PORTS IN_DIG: data_in[0..2]\n'
        '.PORTS OUT: Q_act[0..2]\n'
        '.PORTS OUT: Q_ret[0..2]\n'
        '.MAP clk -> 2\n'
        '.MAP data_in -> 3..5\n'
        '.MAP Q_act -> 7..9\n'
        '.MAP Q_ret -> 11..13\n'
        '.PARAMS V_HIGH=5.0 V_LOW=0.0 R_OUT=50 V_TH=2.5 R_IN=1meg\n'
        '.LANG vhdl\n'
        '.CODE\n'
        'Q_act[0] <= data_in[0];\n'
        'Q_act[1] <= data_in[1];\n'
        'Q_act[2] <= data_in[2];\n'
        'process(clk) begin\n'
        '  if rising_edge(clk) then\n'
        '    Q_ret[0] <= data_in[0];\n'
        '    Q_ret[1] <= data_in[1];\n'
        '    Q_ret[2] <= data_in[2];\n'
        '  end if;\n'
        'end process;\n'
        '.ENDCODE\n'
        '.ENDS\n'
        '\n'
        '* DAC 1 (Entrada - Combinacional)\n'
        'R_a0 7 15 8k\n'
        'R_a1 8 15 4k\n'
        'R_a2 9 15 2k\n'
        'R_fa 15 17 1k\n'
        'E_op1 17 0 0 15 1e6\n'
        'R_ia1 17 18 1k\n'
        'R_ia2 18 19 1k\n'
        'E_in1 19 0 0 18 1e6\n'
        '\n'
        '* DAC 2 (Salida - Retrasada)\n'
        'R_b0 11 20 8k\n'
        'R_b1 12 20 4k\n'
        'R_b2 13 20 2k\n'
        'R_fb 20 22 1k\n'
        'E_op2 22 0 0 20 1e6\n'
        'R_ib1 22 23 1k\n'
        'R_ib2 23 24 1k\n'
        'E_in2 24 0 0 23 1e6\n'
        '\n'
        '.plot V(1) V(2) V(19) V(24) SUBPLOT=ON\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_guardar_drive = widgets.Button(description='Guardar en Drive', button_style='info')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

error_panel = widgets.HTML(value="", layout=widgets.Layout(display='none'))
success_panel = widgets.HTML(value="", layout=widgets.Layout(display='none'))

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    error_panel.layout.display = 'none'
    success_panel.layout.display = 'none'
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_guardar_drive(b):
    if not app_state["drive_mounted"]:
        try:
            drive.mount('/content/drive')
            app_state["drive_mounted"] = True
        except:
            error_panel.value = "<div class='error-panel'>Error al montar Google Drive.</div>"
            error_panel.layout.display = 'block'
            return

    filename = app_state["current_file"] or 'mi_circuito.cir'
    ruta = f'/content/drive/MyDrive/SPICE_Simulator/{filename}'
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w') as f: f.write(editor.value)

    success_panel.value = f"<div class='success-panel'>Guardado exitosamente en Google Drive:<br>{ruta}</div>"
    success_panel.layout.display = 'block'
    error_panel.layout.display = 'none'

def on_simular(b):
    with output_area:
        clear_output()
        error_panel.layout.display = 'none'
        success_panel.layout.display = 'none'

        try:
            c, directivas, errores = parsear_netlist(editor.value)

            if errores:
                html_err = "<div class='error-panel'><b>Advertencias del Parser:</b><br>"
                for e in errores: html_err += f" - {e}<br>"
                html_err += "</div>"
                error_panel.value = html_err
                error_panel.layout.display = 'block'

            if not c.componentes and not c.bloques_codigo:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"

                if directivas['subplot']:
                    fig, axes = plt.subplots(nrows=len(nodos_a_graficar), ncols=1, figsize=(10, 3*len(nodos_a_graficar)), sharex=True)
                    if len(nodos_a_graficar) == 1: axes = [axes]

                    for ax, n in zip(axes, nodos_a_graficar):
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')
                            ax.set_ylabel('Voltaje [V]')
                            ax.grid(True)
                            ax.legend(loc='upper left')

                    axes[-1].set_xlabel('Time [s]')
                    fig.suptitle(titulo)
                else:
                    fig, ax = plt.subplots(figsize=(10, 5))
                    for n in nodos_a_graficar:
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')

                    ax.set_title(titulo)
                    ax.set_xlabel('Time [s]')
                    ax.set_ylabel('Voltage [V]')
                    ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            error_panel.value = "<div class='error-panel'>ERROR: matriz singular.</div>"
            error_panel.layout.display = 'block'
        except Exception as e:
            error_panel.value = f"<div class='error-panel'>ERROR: {e}</div>"
            error_panel.layout.display = 'block'

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_guardar_drive.on_click(on_guardar_drive)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_guardar_drive, btn_simular])
display(barra, uploader, error_panel, success_panel, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

HTML(value='', layout=Layout(display='none'))

HTML(value='', layout=Layout(display='none'))

Textarea(value='* Sistema Completo: ADC (Python) -> VHDL -> 2 DACs + Reloj\nV_in 1 0 PWL(0 0 10m 5 20m 0)\nV_c…

Output()

In [2]:
# =========================================================
#  Simulador SPICE + Microcontroladores (C / Python / VHDL)
#  Version 5.E: Bloque de Funcion de Transferencia (FDT)
#  Una sola celda. Pega y ejecuta.
# =========================================================
import re
import os
import subprocess
import ctypes
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files as colab_files, drive

# ---------------------------------------------------------
# 0. ESTILO CSS
# ---------------------------------------------------------
display(HTML("""
<style>
    .widget-textarea textarea {
        font-size: 16px !important;
        font-family: 'Courier New', monospace !important;
        line-height: 1.4 !important;
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
        border: 1px solid #444 !important;
    }
    .widget-button { font-size: 14px !important; font-weight: bold !important; }
    .error-panel { background-color: #f8d7da; color: #721c24; padding: 10px; border-radius: 5px; border: 1px solid #f5c6cb; font-family: monospace; white-space: pre-wrap; }
    .success-panel { background-color: #d4edda; color: #155724; padding: 10px; border-radius: 5px; border: 1px solid #c3e6cb; font-family: monospace; white-space: pre-wrap; }
</style>
"""))

# ---------------------------------------------------------
# 1. COMPILADOR C AL VUELO
# ---------------------------------------------------------
def compilar_c_y_cargar(nombre, codigo, puertos_in, puertos_out):
    return None

# ---------------------------------------------------------
# 2. MOTOR MNA
# ---------------------------------------------------------
class CircuitoMNA:
    def __init__(self):
        self.nodos = set()
        self.componentes = []
        self.fuentes_v = []
        self.fuentes_pwl = {}
        self.fuentes_pulse = {}
        self.bloques_codigo = []
        self.bloques_fdt = []

    def agregar_nodo(self, n):
        self.nodos.add(n)

    def R(self, nombre, n1, n2, valor):
        if valor == 0: raise ValueError(f"R {nombre} con valor 0")
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('R', nombre, n1, n2, valor))

    def V(self, nombre, n_pos, n_neg, valor, pwl=None, pulse=None):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.fuentes_v.append(nombre)
        self.componentes.append(('V', nombre, n_pos, n_neg, valor))
        if pwl: self.fuentes_pwl[nombre] = pwl
        if pulse: self.fuentes_pulse[nombre] = pulse

    def I(self, nombre, n_pos, n_neg, valor):
        self.agregar_nodo(n_pos); self.agregar_nodo(n_neg)
        self.componentes.append(('I', nombre, n_pos, n_neg, valor))

    def E(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.fuentes_v.append(nombre)
        self.componentes.append(('E', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia))

    def G(self, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm):
        for n in (n_out_p, n_out_n, n_ctrl_p, n_ctrl_n): self.agregar_nodo(n)
        self.componentes.append(('G', nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm))

    def C(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.componentes.append(('C', nombre, n1, n2, valor))

    def L(self, nombre, n1, n2, valor):
        self.agregar_nodo(n1); self.agregar_nodo(n2)
        self.fuentes_v.append(nombre)
        self.componentes.append(('L', nombre, n1, n2, valor))

    def _obtener_valor_v(self, nombre, tiempo):
        if nombre in self.fuentes_pwl:
            puntos = self.fuentes_pwl[nombre]
            if tiempo <= puntos[0][0]: return puntos[0][1]
            if tiempo >= puntos[-1][0]: return puntos[-1][1]
            for i in range(len(puntos) - 1):
                t0, v0 = puntos[i]
                t1, v1 = puntos[i+1]
                if t0 <= tiempo <= t1:
                    return v0 + (v1 - v0) * (tiempo - t0) / (t1 - t0)
        if nombre in self.fuentes_pulse:
            v1, v2, delay, rise, fall, width, period = self.fuentes_pulse[nombre]
            if tiempo < delay: return v1
            t_rel = (tiempo - delay) % period
            if t_rel < rise: return v1 + (v2 - v1) * (t_rel / rise)
            if t_rel < rise + width: return v2
            if t_rel < rise + width + fall: return v2 + (v1 - v2) * ((t_rel - rise - width) / fall)
            return v1
        for comp in self.componentes:
            if comp[0] == 'V' and comp[1] == nombre:
                return comp[4]
        return 0.0

    def agregar_bloque_codigo(self, nombre, pins_dig, pins_analog, pins_out, mapeo, lang, codigo, params):
        def aplanar(lista_pines, mapeo):
            aplanados = []
            nodos = []
            for p in lista_pines:
                if '[' in p:
                    base = p.split('[')[0]
                    rango = p.split('[')[1].replace(']', '')
                    ini, fin = rango.split('..')
                    for i in range(int(ini), int(fin) + 1):
                        nombre_bit = f"{base}_{i}"
                        aplanados.append(nombre_bit)
                        if nombre_bit in mapeo:
                            nodos.append(mapeo[nombre_bit])
                        elif base in mapeo and isinstance(mapeo[base], list):
                            nodos.append(mapeo[base][i - int(ini)])
                else:
                    aplanados.append(p)
                    if p in mapeo:
                        nodos.append(mapeo[p])
            return aplanados, nodos

        dig_vars, nodos_dig = aplanar(pins_dig, mapeo)
        analog_vars, nodos_analog = aplanar(pins_analog, mapeo)
        out_vars, nodos_out = aplanar(pins_out, mapeo)

        for n in nodos_dig + nodos_analog + nodos_out: self.agregar_nodo(n)

        for i, p in enumerate(out_vars):
            nombre_fuente = f"V_{nombre}_{p}"
            self.fuentes_v.append(nombre_fuente)
            self.componentes.append(('V_BLOQUE', nombre_fuente, nodos_out[i], 0, 0.0))

        self.bloques_codigo.append({
            'nombre': nombre,
            'dig_vars': dig_vars,
            'analog_vars': analog_vars,
            'out_vars': out_vars,
            'dig_nodos': nodos_dig,
            'analog_nodos': nodos_analog,
            'out_nodos': nodos_out,
            'lang': lang.lower(),
            'codigo': codigo,
            'v_high': params.get('V_HIGH', 5.0),
            'v_low': params.get('V_LOW', 0.0),
            'r_out': params.get('R_OUT', 50.0),
            'r_in': params.get('R_IN', 1e6),
            'v_th': params.get('V_TH', 2.5),
            'params': params,
            'estado_anterior': {p: params.get('V_LOW', 0.0) for p in out_vars},
            '_prev_clk': 0.0
        })

    def agregar_bloque_fdt(self, nombre, vin_node, vout_node, num, den, params):
        # Validaciones
        if not den or len(den) == 0:
            raise ValueError(f"Bloque {nombre}: FDT sin DEN")
        if den[0] == 0:
            raise ValueError(f"Bloque {nombre}: coeficiente principal de DEN no puede ser 0")
        if len(num) > len(den):
            raise ValueError(f"Bloque {nombre}: FDT impropia (grado NUM > grado DEN)")
        if num and num[0] == 0 and not (len(num) == 1 and num[0] == 0):
            raise ValueError(f"Bloque {nombre}: coeficiente principal de NUM no puede ser 0")

        # Normalizar coeficientes por DEN[0]
        a0 = den[0]
        num = [c / a0 for c in num]
        den = [c / a0 for c in den]

        # Rellenar num con ceros hasta que tenga longitud len(den) - 1 + 1 = len(den)
        num_padded = num + [0.0] * (len(den) - len(num))

        # Grados
        n = len(den) - 1  # grado del denominador
        m = len(num) - 1  # grado del numerador original (antes de rellenar)

        # Construir matrices de estado en forma canonica controlable
        # dx/dt = A*x + B*u ; y = C*x + D*u
        if n == 0:
            # Caso estatico: H(s) = b0/a0
            A = np.zeros((0, 0))
            B = np.zeros(0)
            C = np.zeros(0)
            D = num_padded[-1] if len(num_padded) > 0 else 0.0
            n_estado = 0
        else:
            A = np.zeros((n, n))
            for i in range(n - 1):
                A[i, i + 1] = 1.0
            # Ultima fila: -a_{n-1}/a_n, ..., -a_1/a_n, -a_0/a_n  (ya normalizados)
            for i in range(n):
                A[n - 1, i] = -den[n - i]  # den[0]=1, den[1]=a_{n-1}, ..., den[n]=a_0
            B = np.zeros(n)
            B[n - 1] = 1.0
            # Salida: y = sum b_i * x_i + D*u
            # num_padded tiene len(den) coeficientes: b_m, ..., b_0 (con ceros al inicio)
            # corresponden a x_1, ..., x_n (los indices van en orden)
            C = np.zeros(n)
            for i in range(n):
                C[i] = num_padded[i]  # num_padded[i] corresponde al coeficiente de s^{n-i}
            D = 0.0
            n_estado = n

        self.agregar_nodo(vin_node)
        self.agregar_nodo(vout_node)

        # Registrar fuente de voltaje en el nodo de salida (como los bloques Python/VHDL)
        nombre_fuente = f"V_{nombre}_y"
        self.fuentes_v.append(nombre_fuente)
        self.componentes.append(('V_BLOQUE', nombre_fuente, vout_node, 0, 0.0))

        r_in = params.get('R_IN', 1e6)

        self.bloques_fdt.append({
            'nombre': nombre,
            'vin_node': vin_node,
            'vout_node': vout_node,
            'A': A, 'B': B, 'C': C, 'D': D,
            'n_estado': n_estado,
            'x': np.zeros(n_estado),
            'y': 0.0,
            'r_in': r_in,
            'nombre_fuente': nombre_fuente,
        })

    def _resolver_un_paso(self, modo='dc', dt=None, vc_prev=None, il_prev=None, tiempo_actual=0.0):
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        idx = {n: i for i, n in enumerate(nodos_activos)}
        n_nodos = len(nodos_activos)
        n_fv = len(self.fuentes_v)
        size = n_nodos + n_fv

        A = np.zeros((size, size))
        b = np.zeros(size)
        idx_fv = {nombre: n_nodos + k for k, nombre in enumerate(self.fuentes_v)}

        for comp in self.componentes:
            tipo = comp[0]
            if tipo == 'R':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                g = 1.0 / valor
                if i1 is not None: A[i1, i1] += g
                if i2 is not None: A[i2, i2] += g
                if i1 is not None and i2 is not None:
                    A[i1, i2] -= g; A[i2, i1] -= g
            elif tipo == 'C':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    g = 1e-12
                    if i1 is not None: A[i1, i1] += g
                    if i2 is not None: A[i2, i2] += g
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= g; A[i2, i1] -= g
                else:
                    geq = valor / dt
                    vc = vc_prev.get(nombre, 0.0)
                    ieq = geq * vc
                    if i1 is not None: A[i1, i1] += geq; b[i1] += ieq
                    if i2 is not None: A[i2, i2] += geq; b[i2] -= ieq
                    if i1 is not None and i2 is not None:
                        A[i1, i2] -= geq; A[i2, i1] -= geq
            elif tipo == 'I':
                _, nombre, n1, n2, valor = comp
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: b[i1] -= valor
                if i2 is not None: b[i2] += valor
            elif tipo == 'V':
                _, nombre, n1, n2, valor_const = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                b[k] = self._obtener_valor_v(nombre, tiempo_actual)
            elif tipo == 'V_BLOQUE':
                _, nombre, nodo_out, _, _ = comp
                k = idx_fv[nombre]
                i1 = idx.get(nodo_out)
                if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                b[k] = 0.0
            elif tipo == 'L':
                _, nombre, n1, n2, valor = comp
                k = idx_fv[nombre]
                i1, i2 = idx.get(n1), idx.get(n2)
                if modo == 'dc':
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    b[k] = 0.0
                else:
                    req = valor / dt
                    il = il_prev.get(nombre, 0.0)
                    veq = -req * il
                    if i1 is not None: A[i1, k] += 1; A[k, i1] += 1
                    if i2 is not None: A[i2, k] -= 1; A[k, i2] -= 1
                    A[k, k] -= req; b[k] = veq
            elif tipo == 'G':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, gm = comp
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None and ic_p is not None: A[io_p, ic_p] += gm
                if io_p is not None and ic_n is not None: A[io_p, ic_n] -= gm
                if io_n is not None and ic_p is not None: A[io_n, ic_p] -= gm
                if io_n is not None and ic_n is not None: A[io_n, ic_n] += gm
            elif tipo == 'E':
                _, nombre, n_out_p, n_out_n, n_ctrl_p, n_ctrl_n, ganancia = comp
                k = idx_fv[nombre]
                io_p, io_n = idx.get(n_out_p), idx.get(n_out_n)
                ic_p, ic_n = idx.get(n_ctrl_p), idx.get(n_ctrl_n)
                if io_p is not None: A[io_p, k] += 1; A[k, io_p] += 1
                if io_n is not None: A[io_n, k] -= 1; A[k, io_n] -= 1
                if ic_p is not None: A[k, ic_p] -= ganancia
                if ic_n is not None: A[k, ic_n] += ganancia

        # R_IN de bloques codigo
        for bloque in self.bloques_codigo:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                for nodo_in in bloque['dig_nodos'] + bloque['analog_nodos']:
                    i_node = idx.get(nodo_in)
                    if i_node is not None:
                        A[i_node, i_node] += g_in

        # R_IN de bloques FDT
        for bloque in self.bloques_fdt:
            if bloque['r_in'] > 0:
                g_in = 1.0 / bloque['r_in']
                i_node = idx.get(bloque['vin_node'])
                if i_node is not None:
                    A[i_node, i_node] += g_in

        # Actualizar valores de fuentes de bloques codigo
        for bloque in self.bloques_codigo:
            for i, p in enumerate(bloque['out_vars']):
                nombre_fuente = f"V_{bloque['nombre']}_{p}"
                k = idx_fv[nombre_fuente]
                estado = bloque['estado_anterior'][p]
                b[k] = bloque['v_high'] if estado > 0.5 else bloque['v_low']

        # Actualizar valores de fuentes de bloques FDT
        for bloque in self.bloques_fdt:
            k = idx_fv[bloque['nombre_fuente']]
            b[k] = bloque['y']

        x = np.linalg.solve(A, b)
        voltajes = {n: x[idx[n]] for n in nodos_activos}
        voltajes[0] = 0.0
        corrientes = {f'I_{nombre}': x[k] for nombre, k in idx_fv.items()}
        return voltajes, corrientes

    def _ejecutar_logica_bloques(self, voltajes_actuales, dt):
        for bloque in self.bloques_codigo:
            entradas = {}
            for var in bloque['dig_vars']:
                nodo_in = bloque['dig_nodos'][bloque['dig_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = 1.0 if v_actual > bloque['v_th'] else 0.0
            for var in bloque['analog_vars']:
                nodo_in = bloque['analog_nodos'][bloque['analog_vars'].index(var)]
                v_actual = voltajes_actuales.get(nodo_in, 0.0)
                entradas[var] = v_actual

            if bloque['lang'] == 'python':
                global_env = dict(entradas)
                for p in bloque['out_vars']:
                    global_env[p] = bloque['estado_anterior'].get(p, 0.0)

                for k, v in bloque['params'].items(): global_env[k.lower()] = v
                global_env['v_high'] = bloque['v_high']
                global_env['v_low'] = bloque['v_low']
                global_env['dt'] = dt

                codigo_python = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', bloque['codigo'])

                try:
                    exec(codigo_python, global_env, global_env)
                    for p in bloque['out_vars']:
                        bloque['estado_anterior'][p] = float(global_env[p])
                except Exception as e:
                    pass

            elif bloque['lang'] == 'vhdl':
                res = dict(bloque['estado_anterior'])

                clk_actual = entradas.get('clk', 0.0)
                clk_previo = bloque['_prev_clk']
                flanco_subida = (clk_actual > 0.5) and (clk_previo < 0.5)
                bloque['_prev_clk'] = clk_actual

                local_env = dict(entradas)
                for p in bloque['out_vars']:
                    local_env[p] = bloque['estado_anterior'].get(p, 0.0)

                codigo_limpio = re.sub(r'--.*', '', bloque['codigo'])
                codigo_limpio = re.sub(r'(\w+)\[(\d+)\]', r'\1_\2', codigo_limpio)

                match_process = re.search(r'process\s*\(.*?\)\s*begin(.*?)end\s*process', codigo_limpio, re.IGNORECASE | re.DOTALL)

                if match_process:
                    cuerpo_process = match_process.group(1)
                    cuerpo_process = re.sub(r'if\s+rising_edge\s*\(.*?\)\s*then', '', cuerpo_process, flags=re.IGNORECASE)
                    cuerpo_process = re.sub(r'end\s+if', '', cuerpo_process, flags=re.IGNORECASE)

                    codigo_fuera = codigo_limpio.replace(match_process.group(0), '')

                    for sent in [s.strip() for s in codigo_fuera.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                    if flanco_subida:
                        for sent in [s.strip() for s in cuerpo_process.split(';') if s.strip()]:
                            match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                            if match:
                                out_var = match.group(1).strip()
                                expr = match.group(2).strip().rstrip(';')
                                sent_eval = expr.replace("'", "")
                                sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                                sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                                try: res[out_var] = float(eval(sent_eval, {}, local_env))
                                except: pass
                else:
                    for sent in [s.strip() for s in codigo_limpio.split(';') if s.strip()]:
                        match = re.match(r'(\w+)\s*<=\s*(.*)', sent, re.IGNORECASE)
                        if match:
                            out_var = match.group(1).strip()
                            expr = match.group(2).strip().rstrip(';')
                            sent_eval = expr.replace("'", "")
                            sent_eval = re.sub(r'\bAND\b', ' and ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bOR\b', ' or ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bNOT\b', ' not ', sent_eval, flags=re.IGNORECASE)
                            sent_eval = re.sub(r'\bXOR\b', ' ^ ', sent_eval, flags=re.IGNORECASE)
                            try: res[out_var] = float(eval(sent_eval, {}, local_env))
                            except: pass

                for p in bloque['out_vars']:
                    bloque['estado_anterior'][p] = res.get(p, 0.0)

        # Actualizar bloques FDT con Euler hacia atras
        if dt is not None and dt > 0:
            for bloque in self.bloques_fdt:
                if bloque['n_estado'] == 0:
                    # Estatico: y = D * u
                    u = voltajes_actuales.get(bloque['vin_node'], 0.0)
                    bloque['y'] = bloque['D'] * u
                else:
                    # u es el voltaje de entrada
                    u = voltajes_actuales.get(bloque['vin_node'], 0.0)
                    # Euler atras: x_nuevo = x_prev + dt * (A*x_nuevo + B*u)
                    # => (I - dt*A) * x_nuevo = x_prev + dt*B*u
                    I = np.eye(bloque['n_estado'])
                    M = I - dt * bloque['A']
                    rhs = bloque['x'] + dt * bloque['B'] * u
                    x_nuevo = np.linalg.solve(M, rhs)
                    # y = C*x + D*u
                    y_nuevo = bloque['C'] @ x_nuevo + bloque['D'] * u
                    bloque['x'] = x_nuevo
                    bloque['y'] = y_nuevo

    def punto_operacion(self):
        # Inicializar estados de FDT en cero
        for bloque in self.bloques_fdt:
            bloque['x'] = np.zeros(bloque['n_estado'])
            bloque['y'] = 0.0
        for _ in range(10):
            voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
            self._ejecutar_logica_bloques(voltajes, None)

        voltajes, corrientes = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)

        vc0 = {}
        il0 = {}
        for comp in self.componentes:
            if comp[0] == 'C':
                _, nombre, n1, n2, _ = comp
                vc0[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
            elif comp[0] == 'L':
                _, nombre, n1, n2, _ = comp
                il0[nombre] = corrientes[f'I_{nombre}']
        return {'voltajes': voltajes, 'corrientes': corrientes, 'vc0': vc0, 'il0': il0}

    def transitorio(self, tstop, dt, uic=False):
        if uic:
            vc_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'C'}
            il_prev = {c[1]: 0.0 for c in self.componentes if c[0] == 'L'}
            # Reiniciar estados de FDT
            for bloque in self.bloques_fdt:
                bloque['x'] = np.zeros(bloque['n_estado'])
                bloque['y'] = 0.0
        else:
            op = self.punto_operacion()
            vc_prev = dict(op['vc0'])
            il_prev = dict(op['il0'])

        n_pasos = int(np.ceil(tstop / dt)) + 1
        tiempos = np.arange(n_pasos) * dt
        nodos_activos = sorted(n for n in self.nodos if n != 0)
        historial = {n: np.zeros(n_pasos) for n in nodos_activos}

        voltajes, _ = self._resolver_un_paso(modo='dc', tiempo_actual=0.0)
        for n in nodos_activos: historial[n][0] = voltajes.get(n, 0.0)

        for paso in range(1, n_pasos):
            t_actual = tiempos[paso]
            voltajes, corrientes = self._resolver_un_paso(
                modo='tran', dt=dt, vc_prev=vc_prev, il_prev=il_prev, tiempo_actual=t_actual
            )

            for n in nodos_activos:
                historial[n][paso] = voltajes.get(n, 0.0)

            self._ejecutar_logica_bloques(voltajes, dt)

            for comp in self.componentes:
                if comp[0] == 'C':
                    _, nombre, n1, n2, _ = comp
                    vc_prev[nombre] = voltajes.get(n1, 0.0) - voltajes.get(n2, 0.0)
                elif comp[0] == 'L':
                    _, nombre, n1, n2, _ = comp
                    il_prev[nombre] = corrientes[f'I_{nombre}']

        return tiempos, historial

# ---------------------------------------------------------
# 3. PARSER DE NETLIST
# ---------------------------------------------------------
def sufijo_a_valor(token):
    token = token.strip().lower()
    match = re.match(r'^([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([a-z]*)$', token)
    if not match: raise ValueError(f"Valor no reconocido: {token}")
    num = float(match.group(1)); suf = match.group(2)
    escalas = {'t': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3, 'm': 1e-3, 'u': 1e-6, 'n': 1e-9, 'p': 1e-12, 'f': 1e-15}
    if suf == '': return num
    for clave in sorted(escalas, key=len, reverse=True):
        if suf.startswith(clave): return num * escalas[clave]
    raise ValueError(f"Sufijo no reconocido: {suf}")

def parsear_lista_coeficientes(s):
    s = s.strip()
    if s.startswith('['): s = s[1:]
    if s.endswith(']'): s = s[:-1]
    if not s: return []
    return [sufijo_a_valor(x) for x in s.split(',')]

def parsear_netlist(texto):
    c = CircuitoMNA()
    errores = []
    directivas = {'tran': None, 'op': False, 'uic': False, 'plot': [], 'subplot': False}

    en_subckt = False
    subckt_nombre = ""
    pins_dig = []
    pins_analog = []
    pins_out = []
    subckt_lang = "python"
    subckt_codigo = []
    subckt_mapeo = {}
    subckt_params = {}
    subckt_fdt_num = None
    subckt_fdt_den = None

    for n_linea, linea_original in enumerate(texto.splitlines(), start=1):
        linea = linea_original.strip()
        if not linea or linea.startswith('*'): continue

        if en_subckt:
            tokens = linea.split()
            d = tokens[0].lower()

            if d == '.code': continue
            elif d == '.endcode': continue
            elif d == '.map':
                tokens_clean = linea.split()
                if len(tokens_clean) >= 4:
                    var = tokens_clean[1]
                    if tokens_clean[2] == '->':
                        val = tokens_clean[3]
                        if '..' in val:
                            ini, fin = val.split('..')
                            subckt_mapeo[var] = list(range(int(ini), int(fin)+1))
                        else:
                            subckt_mapeo[var] = int(val)
                    else:
                        errores.append(f"L{n_linea}: Formato .MAP invalido")
            elif d == '.params':
                for t in tokens[1:]:
                    if '=' in t:
                        k, v = t.split('=')
                        try:
                            subckt_params[k] = sufijo_a_valor(v)
                        except:
                            pass
            elif d == '.ports':
                tipo_puerto = tokens[1].upper().replace(':', '')
                for p in tokens[2:]:
                    if tipo_puerto == 'IN_DIG': pins_dig.append(p)
                    elif tipo_puerto == 'IN_ANALOG': pins_analog.append(p)
                    elif tipo_puerto == 'OUT': pins_out.append(p)
            elif d == '.lang':
                subckt_lang = tokens[1]
            elif d == '.fdt':
                # .FDT NUM: [...]  o .FDT DEN: [...]
                match = re.match(r'\.fdt\s+(num|den)\s*:\s*(.*)', linea, re.IGNORECASE)
                if match:
                    tipo_fdt = match.group(1).lower()
                    coefs_str = match.group(2).strip()
                    try:
                        coefs = parsear_lista_coeficientes(coefs_str)
                    except Exception as e:
                        errores.append(f"L{n_linea}: Error en coeficientes FDT: {e}")
                        coefs = None
                    if coefs is not None:
                        if tipo_fdt == 'num':
                            subckt_fdt_num = coefs
                        else:
                            subckt_fdt_den = coefs
                else:
                    errores.append(f"L{n_linea}: Formato .FDT invalido (use '.FDT NUM: [...]' o '.FDT DEN: [...]')")
            elif d == '.ends':
                if subckt_fdt_den is not None:
                    # Es un bloque FDT
                    if subckt_fdt_num is None:
                        errores.append(f"Bloque {subckt_nombre}: falta .FDT NUM")
                    elif not pins_analog or not pins_out:
                        errores.append(f"Bloque {subckt_nombre}: FDT requiere IN_ANALOG y OUT")
                    else:
                        # Obtener nodos mapeados
                        vin_var = pins_analog[0]
                        vout_var = pins_out[0]
                        if vin_var not in subckt_mapeo or vout_var not in subckt_mapeo:
                            errores.append(f"Bloque {subckt_nombre}: falta mapear {vin_var} o {vout_var}")
                        else:
                            vin_node = subckt_mapeo[vin_var]
                            vout_node = subckt_mapeo[vout_var]
                            try:
                                c.agregar_bloque_fdt(
                                    subckt_nombre, vin_node, vout_node,
                                    subckt_fdt_num, subckt_fdt_den, subckt_params
                                )
                            except Exception as e:
                                errores.append(f"Bloque {subckt_nombre}: {e}")
                else:
                    # Es un bloque de codigo normal
                    c.agregar_bloque_codigo(
                        subckt_nombre, pins_dig, pins_analog, pins_out, subckt_mapeo,
                        subckt_lang, "\n".join(subckt_codigo), subckt_params
                    )
                en_subckt = False
            else:
                subckt_codigo.append(linea_original)
            continue

        if linea.lower().startswith('.subckt'):
            en_subckt = True
            subckt_nombre = linea.split()[1]
            pins_dig = []; pins_analog = []; pins_out = []
            subckt_lang = "python"; subckt_codigo = []; subckt_mapeo = {}
            subckt_params = {}
            subckt_fdt_num = None
            subckt_fdt_den = None
            continue

        if linea.startswith('.'):
            tokens = linea.split()
            d = tokens[0].lower()
            if d == '.tran':
                tstep = sufijo_a_valor(tokens[1]); tstop = sufijo_a_valor(tokens[2])
                uic = any(t.lower() == 'uic' for t in tokens[3:])
                directivas['tran'] = (tstep, tstop); directivas['uic'] = uic
            elif d == '.op': directivas['op'] = True
            elif d == '.plot':
                for t in tokens[1:]:
                    if 'SUBPLOT=ON' in t.upper():
                        directivas['subplot'] = True
                    elif 'SUBPLOT=OFF' in t.upper():
                        directivas['subplot'] = False
                    else:
                        match = re.match(r'V\((.*?)\)', t, re.IGNORECASE)
                        if match:
                            try:
                                nodo = int(match.group(1))
                                directivas['plot'].append(nodo)
                            except ValueError:
                                pass
            elif d == '.end': pass
            else: errores.append(f"L{n_linea}: directiva '{d}' no soportada")
            continue

        tokens = linea.split()
        tipo = tokens[0][0].upper()
        try:
            if tipo in ('R', 'V', 'I', 'C', 'L'):
                nombre = tokens[0]
                n1, n2 = int(tokens[1]), int(tokens[2])

                if tipo == 'V' and len(tokens) > 3:
                    token_valor = tokens[3].upper()
                    if token_valor.startswith('PWL('):
                        match_pwl = re.search(r'PWL\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pwl:
                            contenido = match_pwl.group(1).strip()
                            pwl_tokens = contenido.replace(',', ' ').split()
                            pwl_puntos = []
                            for i in range(0, len(pwl_tokens), 2):
                                if i+1 < len(pwl_tokens):
                                    t_val = sufijo_a_valor(pwl_tokens[i])
                                    v_val = sufijo_a_valor(pwl_tokens[i+1])
                                    pwl_puntos.append((t_val, v_val))
                            c.V(nombre, n1, n2, 0.0, pwl=pwl_puntos)
                        else:
                            raise ValueError("Formato PWL invalido")
                    elif token_valor.startswith('PULSE('):
                        match_pulse = re.search(r'PULSE\s*\((.*?)\)', linea, re.IGNORECASE)
                        if match_pulse:
                            contenido = match_pulse.group(1).strip()
                            pulse_tokens = contenido.replace(',', ' ').split()
                            vals = [sufijo_a_valor(t) for t in pulse_tokens]
                            while len(vals) < 7: vals.append(0.0)
                            c.V(nombre, n1, n2, 0.0, pulse=tuple(vals[:7]))
                        else:
                            raise ValueError("Formato PULSE invalido")
                    else:
                        valor = sufijo_a_valor(tokens[3])
                        getattr(c, tipo)(nombre, n1, n2, valor)
                else:
                    valor = sufijo_a_valor(tokens[3])
                    getattr(c, tipo)(nombre, n1, n2, valor)
            elif tipo in ('E', 'G'):
                nombre = tokens[0]
                nop, non = int(tokens[1]), int(tokens[2])
                ncp, ncn = int(tokens[3]), int(tokens[4])
                valor = sufijo_a_valor(tokens[5])
                getattr(c, tipo)(nombre, nop, non, ncp, ncn, valor)
            else:
                errores.append(f"L{n_linea}: tipo '{tipo}' no soportado")
        except Exception as e:
            errores.append(f"L{n_linea}: {e}")
    return c, directivas, errores

# ---------------------------------------------------------
# 4. GUI
# ---------------------------------------------------------
app_state = {"current_file": None, "drive_mounted": False}

editor = widgets.Textarea(
    value=(
        '* Prueba de FDT: Filtro Pasa-Bajos de 1er Orden\n'
        'V_in 1 0 PWL(0 0 1m 1 20m 1)\n'
        '\n'
        '.SUBCKT filtro_pb\n'
        '.PORTS IN_ANALOG: vin\n'
        '.PORTS OUT: vout\n'
        '.MAP vin -> 1\n'
        '.MAP vout -> 2\n'
        '.FDT NUM: [1]\n'
        '.FDT DEN: [0.002, 1]\n'
        '.ENDS\n'
        '\n'
        'R_load 2 0 1meg\n'
        '\n'
        '.plot V(1) V(2) SUBPLOT=ON\n'
        '.tran 10u 20m uic\n'
        '.end\n'
    ),
    description='Netlist:',
    layout=widgets.Layout(width='100%', height='450px')
)

btn_nuevo = widgets.Button(description='Nuevo', button_style='info')
btn_abrir = widgets.Button(description='Abrir', button_style='primary')
btn_guardar = widgets.Button(description='Guardar', button_style='success')
btn_guardar_como = widgets.Button(description='Guardar como...', button_style='warning')
btn_guardar_drive = widgets.Button(description='Guardar en Drive', button_style='info')
btn_simular = widgets.Button(description='Simular', button_style='danger')

output_area = widgets.Output()

error_panel = widgets.HTML(value="", layout=widgets.Layout(display='none'))
success_panel = widgets.HTML(value="", layout=widgets.Layout(display='none'))

uploader = widgets.FileUpload(
    accept='.cir,.net,.sp,.txt,.vhd', multiple=False,
    description='Subir archivo SPICE',
    layout=widgets.Layout(display='none')
)

def on_nuevo(b):
    editor.value = ''; app_state["current_file"] = None
    error_panel.layout.display = 'none'
    success_panel.layout.display = 'none'
    with output_area: clear_output(); print("Editor limpio.")

def on_abrir(b): uploader.layout.display = 'flex'

def on_file_uploaded(change):
    if not change.new: return
    for filename, fileinfo in change.new.items():
        editor.value = fileinfo['content'].decode('utf-8')
        app_state["current_file"] = filename
        with output_area: clear_output(); print(f"Cargado: {filename}")
    uploader.layout.display = 'none'

def on_guardar(b):
    filename = app_state["current_file"] or 'circuito_temporal.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    with output_area: clear_output(); print(f"Guardado en /content/{filename}")

def on_guardar_como(b):
    filename = app_state["current_file"] or 'mi_circuito.cir'
    with open(f'/content/{filename}', 'w') as f: f.write(editor.value)
    colab_files.download(f'/content/{filename}')

def on_guardar_drive(b):
    if not app_state["drive_mounted"]:
        try:
            drive.mount('/content/drive')
            app_state["drive_mounted"] = True
        except:
            error_panel.value = "<div class='error-panel'>Error al montar Google Drive.</div>"
            error_panel.layout.display = 'block'
            return

    filename = app_state["current_file"] or 'mi_circuito.cir'
    ruta = f'/content/drive/MyDrive/SPICE_Simulator/{filename}'
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w') as f: f.write(editor.value)

    success_panel.value = f"<div class='success-panel'>Guardado exitosamente en Google Drive:<br>{ruta}</div>"
    success_panel.layout.display = 'block'
    error_panel.layout.display = 'none'

def on_simular(b):
    with output_area:
        clear_output()
        error_panel.layout.display = 'none'
        success_panel.layout.display = 'none'

        try:
            c, directivas, errores = parsear_netlist(editor.value)

            if errores:
                html_err = "<div class='error-panel'><b>Advertencias del Parser:</b><br>"
                for e in errores: html_err += f" - {e}<br>"
                html_err += "</div>"
                error_panel.value = html_err
                error_panel.layout.display = 'block'

            if not c.componentes and not c.bloques_codigo and not c.bloques_fdt:
                print("No hay componentes."); return

            if directivas['tran'] is not None:
                tstep, tstop = directivas['tran']; uic = directivas['uic']

                tiempos, historial = c.transitorio(tstop, tstep, uic=uic)

                nodos_a_graficar = directivas['plot']
                if not nodos_a_graficar:
                    nodos_a_graficar = sorted(historial.keys())

                titulo = f"Transient (tstep: {tstep:.3g}s, tstop: {tstop:.3g}s, uic: {uic})"

                if directivas['subplot']:
                    fig, axes = plt.subplots(nrows=len(nodos_a_graficar), ncols=1, figsize=(10, 3*len(nodos_a_graficar)), sharex=True)
                    if len(nodos_a_graficar) == 1: axes = [axes]

                    for ax, n in zip(axes, nodos_a_graficar):
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')
                            ax.set_ylabel('Voltaje [V]')
                            ax.grid(True)
                            ax.legend(loc='upper left')

                    axes[-1].set_xlabel('Time [s]')
                    fig.suptitle(titulo)
                else:
                    fig, ax = plt.subplots(figsize=(10, 5))
                    for n in nodos_a_graficar:
                        if n in historial and n != 0:
                            ax.plot(tiempos, historial[n], label=f'V({n})')

                    ax.set_title(titulo)
                    ax.set_xlabel('Time [s]')
                    ax.set_ylabel('Voltage [V]')
                    ax.grid(True); ax.legend(loc='upper left')

                plt.tight_layout(); plt.show()
            else:
                op = c.punto_operacion()
                print("=== DC (Punto de Operacion) ===")
                for b in c.bloques_codigo:
                    print(f"  - {b['nombre']}: In={b['analog_vars']+b['dig_vars']}, Out={b['out_vars']}, Estado={b['estado_anterior']}")

                for n in sorted(op['voltajes'].keys()):
                    if n == 0: continue
                    print(f"  V({n}) = {op['voltajes'][n]:.6g} V")
                for k, v in op['corrientes'].items(): print(f"  {k} = {v:.6g} A")
        except np.linalg.LinAlgError:
            error_panel.value = "<div class='error-panel'>ERROR: matriz singular.</div>"
            error_panel.layout.display = 'block'
        except Exception as e:
            error_panel.value = f"<div class='error-panel'>ERROR: {e}</div>"
            error_panel.layout.display = 'block'

btn_nuevo.on_click(on_nuevo); btn_abrir.on_click(on_abrir)
uploader.observe(on_file_uploaded, names='value')
btn_guardar.on_click(on_guardar); btn_guardar_como.on_click(on_guardar_como)
btn_guardar_drive.on_click(on_guardar_drive)
btn_simular.on_click(on_simular)

barra = widgets.HBox([btn_nuevo, btn_abrir, btn_guardar, btn_guardar_como, btn_guardar_drive, btn_simular])
display(barra, uploader, error_panel, success_panel, editor, output_area)

FileUpload(value={}, accept='.cir,.net,.sp,.txt,.vhd', description='Subir archivo SPICE', layout=Layout(displa…

HTML(value='', layout=Layout(display='none'))

HTML(value='', layout=Layout(display='none'))

Textarea(value='* Prueba de FDT: Filtro Pasa-Bajos de 1er Orden\nV_in 1 0 PWL(0 0 1m 1 20m 1)\n\n.SUBCKT filtr…

Output()